## Step 1 — Runtime check

In [ ]:
import os, sys, platform, multiprocessing, subprocess, shutil
IN_COLAB = 'google.colab' in sys.modules
print('Python  :', sys.version.split()[0], '| Colab:', IN_COLAB)
print('CPU     :', multiprocessing.cpu_count(), 'cores')
print('RAM     : %.1f GB' % (os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9))
print('Disk    : %.1f GB free in /content' % (shutil.disk_usage('/content' if os.path.exists('/content') else '/').free / 1e9))
try:
    out = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.strip()
except FileNotFoundError:
    out = ''
print('GPU     :', out or 'none (fine: the pipeline is CPU-only)')
if sys.version_info < (3, 9):
    raise SystemExit('Python >= 3.9 is required (dict union operator is used in run_seg.py).')

Python  : 3.13.18 | Colab: True
CPU     : 2 cores
RAM     : 13.6 GB
Disk    : 115.6 GB free in /content
GPU     : none (fine: the pipeline is CPU-only)


## Step 2 — Settings and Google Drive
Everything lives in **one** Drive folder: `MyDrive/DOLCHID`. The raw data stay in `cbct_image/`, `cbct_label/`, `hist_*`; everything that must survive a disconnection (extracted data, caches, weights, results, figures, job logs) is written to `WORK`, which is that same folder.


In [ ]:
#@title Settings  (all paths are defined here, nowhere else)
USE_DRIVE = True                 #@param {type:"boolean"}
DATASET_FOLDER = "DOLCHID"       #@param {type:"string"}   # your folder in My Drive (the ONLY Drive folder used)
OUTPUT_SUBFOLDER = ""            #@param {type:"string"}   # "" = write results directly inside DOLCHID; or e.g. "PAL_output"
CASE_PREFIX = "AME"             #@param {type:"string"}   # main study group: AME (your data), RC, DC, KCOT, or "auto"
PARALLEL_JOBS = 2                #@param {type:"integer"}
COPY_DATA_TO_LOCAL_DISK = True   #@param {type:"boolean"}  # copy extracted .npz to /content for fast reads

import os, multiprocessing

if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_ROOT = f'/content/drive/MyDrive/{DATASET_FOLDER}'
else:
    DATA_ROOT = os.path.abspath(DATASET_FOLDER)
assert os.path.isdir(DATA_ROOT), f'{DATA_ROOT} not found - check DATASET_FOLDER'

# raw data (read only)
CBCT_IMG = f'{DATA_ROOT}/cbct_image'    # <ID>_CBCT_Image.nii.gz
CBCT_LBL = f'{DATA_ROOT}/cbct_label'    # <ID>_CBCT_Label.nii.gz
HIST_IMG = f'{DATA_ROOT}/hist_image'
HIST_LBL = f'{DATA_ROOT}/hist_label'

# which patient group is the main study group (RC_*, AME_*, ...)
import re, glob, collections
if CASE_PREFIX == 'auto':
    _pre = [re.match(r'([A-Za-z]+)_\d+', os.path.basename(f)) for f in glob.glob(f'{CBCT_IMG}/*_CBCT_Image.nii*')]
    _cnt = collections.Counter(m.group(1) for m in _pre if m)
    print('patient groups found in cbct_image:', dict(_cnt))
    CASE_PREFIX = 'RC' if _cnt.get('RC') else (_cnt.most_common(1)[0][0] if _cnt else 'RC')

# everything the pipeline writes goes into the same DOLCHID folder
WORK = f'{DATA_ROOT}/{OUTPUT_SUBFOLDER}'.rstrip('/') if OUTPUT_SUBFOLDER else DATA_ROOT
CODE = '/content/pal_code' if os.path.exists('/content') else os.path.abspath('pal_code')   # local, not on Drive

# only the folders that are needed before any script runs; the scripts create the rest when they need it
for d in ('data', 'cache', 'results/tables'):
    os.makedirs(f'{WORK}/{d}', exist_ok=True)
os.makedirs(f'{CODE}/pal', exist_ok=True)

NTH_PER_JOB = max(1, multiprocessing.cpu_count() // max(1, PARALLEL_JOBS))

for k in ('PAL_DOLCHID_URL', 'PAL_DOLCHID_ZIP', 'SMOKE'):
    os.environ.pop(k, None)
os.environ['PAL_WORK']        = WORK
os.environ['PAL_DATA']        = f'{WORK}/data'     # extracted sections (<prefix>_*.npz)
os.environ['PAL_PREFIX']      = CASE_PREFIX        # main study group
os.environ['PAL_DOLCHID_DIR'] = DATA_ROOT
os.environ['PAL_IMAGE_DIR']   = CBCT_IMG
os.environ['PAL_LABEL_DIR']   = CBCT_LBL

print('DATA_ROOT =', DATA_ROOT)
print('CASE_PREFIX =', CASE_PREFIX)
print('WORK      =', WORK, '(same folder)' if WORK == DATA_ROOT else '')
print('CODE      =', CODE)
print('parallel jobs =', PARALLEL_JOBS, '| threads per job =', NTH_PER_JOB)
for p in (CBCT_IMG, CBCT_LBL, HIST_IMG, HIST_LBL):
    print(('OK      ' if os.path.isdir(p) else 'MISSING '), p)


Mounted at /content/drive
DATA_ROOT = /content/drive/MyDrive/DOLCHID
CASE_PREFIX = AME
WORK      = /content/drive/MyDrive/DOLCHID (same folder)
CODE      = /content/pal_code
parallel jobs = 2 | threads per job = 1
OK       /content/drive/MyDrive/DOLCHID/cbct_image
OK       /content/drive/MyDrive/DOLCHID/cbct_label
OK       /content/drive/MyDrive/DOLCHID/hist_image
OK       /content/drive/MyDrive/DOLCHID/hist_label


## Step 3 — Write the source code
Run all cells in this section. Each cell writes one file of the authors' package (paths already adapted for Colab). If you change a file here, re-run its cell.

`pal/__init__.py` — package marker

In [ ]:
%%writefile /content/pal_code/pal/__init__.py
# package marker

Writing /content/pal_code/pal/__init__.py


`pal/core.py` — core library: data, scribble simulation, PSF/Wiener, SPCNN, graph construction, dual GCN, prior correction, ConvCRF, U-Nets, baselines, metrics

In [ ]:
%%writefile /content/pal_code/pal/core.py
"""
Core library for the revised study:
"Scribble-conditioned graph convolutional segmentation of radicular cysts on CBCT
 with a learned pre-filter and pixel-level refinement" (revision of
 "An Enhanced Deep Learning Framework for Periapical Lesion Segmentation and Detection in CBCT Images").

Everything here runs on CPU (PyTorch). Pixel spacing of DOLCHID is 0.3 mm (isotropic).
"""
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import os, glob, ast, time, math
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from scipy import ndimage as ndi
from skimage.segmentation import slic, random_walker
from skimage.morphology import skeletonize, disk
from scipy.ndimage import binary_erosion as _be
import warnings; warnings.filterwarnings('ignore')
def binary_erosion(a, fp):
    return _be(a, structure=fp)
from skimage import restoration, filters, graph as skgraph
from skimage.measure import label as cclabel, regionprops

PIX_MM = 0.3
DATA = _os_pal.environ.get('PAL_DATA', PAL_WORK + '/data')
PREFIX = _os_pal.environ.get('PAL_PREFIX', 'RC')   # main study group
torch.set_num_threads(int(_os_pal.environ.get('NTH', '2')))


# ----------------------------------------------------------------------------------------------
# data
# ----------------------------------------------------------------------------------------------
def load_dataset():
    """Returns dict with arrays for the 216 RC sections (162 lesion + 54 lesion-free)."""
    files = sorted(glob.glob(f'{DATA}/{PREFIX}_*.npz'), key=lambda s: int(s.split('_')[-1][:-4]))
    imgs, msks, pids, secs = [], [], [], []
    extra = {}
    for f in files:
        d = np.load(f)
        pid = os.path.basename(f)[:-4]
        for s in range(4):
            imgs.append(d['img'][s]); msks.append(d['msk'][s].astype(np.uint8)); pids.append(pid); secs.append(s)
        extra[pid] = f
    imgs = np.stack(imgs).astype(np.float32); msks = np.stack(msks)
    pids = np.array(pids); secs = np.array(secs)
    is_les = msks.reshape(len(msks), -1).any(1)
    return dict(img=imgs, msk=msks, pid=pids, sec=secs, lesion=is_les, files=extra)


def size_groups(ds):
    """Patients split at the median per-section lesion area (mm^2) of lesion sections."""
    area = ds['msk'].reshape(len(ds['msk']), -1).sum(1) * PIX_MM ** 2
    les = ds['lesion']
    thr = float(np.median(area[les]))
    grp = np.where(area >= thr, 'large', 'small')
    pat_area = {p: area[(ds['pid'] == p) & les].mean() for p in np.unique(ds['pid'])}
    pat_thr = float(np.median(list(pat_area.values())))
    pat_grp = {p: ('large' if a >= pat_thr else 'small') for p, a in pat_area.items()}
    return area, grp, thr, pat_grp, pat_thr


def make_folds(ds, k=4, seed=0):
    """Patient-level folds stratified by patient lesion-size group."""
    _, _, _, pat_grp, _ = size_groups(ds)
    rng = np.random.default_rng(seed)
    fold_of = {}
    for g in ('small', 'large'):
        ps = sorted([p for p, gg in pat_grp.items() if gg == g], key=lambda s: int(s.split('_')[1]))
        rng.shuffle(ps)
        for i, p in enumerate(ps):
            fold_of[p] = i % k
    return fold_of


def split_train_val(train_pids, seed=0, frac=0.15):
    rng = np.random.default_rng(seed + 99)
    ps = np.array(sorted(train_pids)); rng.shuffle(ps)
    nv = max(1, int(round(frac * len(ps))))
    return list(ps[nv:]), list(ps[:nv])


# ----------------------------------------------------------------------------------------------
# scribble simulation (default = protocol of the original manuscript) and imperfect variants
# ----------------------------------------------------------------------------------------------
def _bfs_stroke(skel_pts_set, start, n):
    out = [start]; seen = {start}; q = [start]
    while q and len(out) < n:
        r, c = q.pop(0)
        for dr in (-1, 0, 1):
            for dc in (-1, 0, 1):
                nb = (r + dr, c + dc)
                if nb in skel_pts_set and nb not in seen:
                    seen.add(nb); out.append(nb); q.append(nb)
                    if len(out) >= n:
                        break
            if len(out) >= n:
                break
    return out


def _strokes_on(region, rng, density, n_strokes, erode_r):
    if region.sum() == 0:
        return []
    e = region
    for r in range(erode_r, -1, -1):
        e = binary_erosion(region, disk(r)) if r > 0 else region
        if e.sum() > 0:
            break
    sk = skeletonize(e)
    pts = list(map(tuple, np.argwhere(sk)))
    if not pts:
        pts = list(map(tuple, np.argwhere(e)))
    n_keep = max(2, int(round(density * len(pts))))
    per = max(1, n_keep // n_strokes)
    S = set(pts); strokes = []
    for _ in range(n_strokes):
        st = pts[rng.integers(len(pts))]
        strokes.append(_bfs_stroke(S, st, per))
    return strokes


def bg_region(mask, ref_mask=None):
    """Surrounding-tissue band where the operator marks background."""
    m = mask if mask.any() else (ref_mask if ref_mask is not None else mask)
    if not m.any():
        m = np.zeros_like(mask, bool); m[118:138, 118:138] = True
    m = m.astype(bool)
    outer = ndi.binary_dilation(m, iterations=30); inner = ndi.binary_dilation(m, iterations=6)
    return outer & ~inner


def simulate_scribbles(mask, rng, mode='default', density=0.15, ref_mask=None):
    """Return int8 map: -1 = unlabelled, 0 = background scribble, 1 = lesion scribble.
    modes: default | boundary | mislabel10 | mislabel20 | click | missing | opA | opB | opC"""
    mask = mask.astype(bool)
    H, W = mask.shape
    lab = np.full((H, W), -1, np.int8)
    bgr = bg_region(mask, ref_mask)
    strokes = []  # (points, class)

    def add(stks, cls):
        for s in stks:
            strokes.append((s, cls))

    if mode in ('default', 'mislabel10', 'mislabel20', 'missing'):
        add(_strokes_on(mask, rng, density, 2, 2), 1)
        add(_strokes_on(bgr, rng, density, 2, 4), 0)
        if mode == 'missing':
            strokes = [s for s in strokes if s[1] == 0]
        if mode.startswith('mislabel'):
            p = 0.10 if mode == 'mislabel10' else 0.20
            strokes = [(s, (1 - c) if rng.random() < p else c) for s, c in strokes]
    elif mode == 'density05':
        add(_strokes_on(mask, rng, 0.05, 2, 2), 1); add(_strokes_on(bgr, rng, 0.05, 2, 4), 0)
    elif mode == 'density30':
        add(_strokes_on(mask, rng, 0.30, 2, 2), 1); add(_strokes_on(bgr, rng, 0.30, 2, 4), 0)
    elif mode == 'boundary':
        # strokes drawn along the lesion boundary, shifted by a random 2-3 px: about half the points
        # fall outside the lesion but are still labelled lesion (and vice-versa for background)
        if mask.any():
            bnd = mask & ~binary_erosion(mask, disk(1))
            add([_shift(s, rng, H, W) for s in _strokes_on(bnd, rng, 0.25, 2, 0)], 1)
            ring = ndi.binary_dilation(mask, iterations=4) & ~ndi.binary_dilation(mask, iterations=3)
            add([_shift(s, rng, H, W) for s in _strokes_on(ring, rng, 0.25, 2, 0)], 0)
        else:
            add(_strokes_on(bgr, rng, density, 2, 4), 0)
    elif mode == 'click':
        if mask.any():
            dt = ndi.distance_transform_edt(mask)
            cand = np.argwhere(dt >= max(1, 0.5 * dt.max()))
            r, c = cand[rng.integers(len(cand))]
            strokes.append(([(r, c)], 1))
        cand = np.argwhere(bgr)
        if len(cand):
            r, c = cand[rng.integers(len(cand))]
            strokes.append(([(r, c)], 0))
    elif mode in ('opA', 'opB', 'opC'):
        # three simulated operator styles (NOT a human study)
        if mode == 'opA':
            add(_strokes_on(mask, rng, 0.10, 1, 2), 1); add(_strokes_on(bgr, rng, 0.10, 1, 4), 0)
        elif mode == 'opB':
            add(_strokes_on(mask, rng, 0.25, 3, 1), 1); add(_strokes_on(bgr, rng, 0.25, 3, 3), 0)
        else:
            if mask.any():
                dt = ndi.distance_transform_edt(mask); cand = np.argwhere(dt >= max(1, 0.5 * dt.max()))
                r, c = cand[rng.integers(len(cand))]; strokes.append(([(r, c)], 1))
            add(_strokes_on(bgr, rng, 0.15, 1, 4), 0)
        # 10 % of strokes are drawn 3 px off target (may cross the boundary)
        strokes = [(_shift(s, rng, H, W, 3) if rng.random() < 0.10 else s, c) for s, c in strokes]
    else:
        raise ValueError(mode)
    for s, c in strokes:
        for (r, cc) in s:
            lab[max(0, r - 1):r + 2, max(0, cc - 1):cc + 2] = c  # 3-px brush
    return lab


def _shift(s, rng, H, W, mag=None):
    mag = mag if mag is not None else rng.integers(2, 4)
    ang = rng.uniform(0, 2 * np.pi)
    dr, dc = int(round(mag * np.sin(ang))), int(round(mag * np.cos(ang)))
    return [(int(np.clip(r + dr, 0, H - 1)), int(np.clip(c + dc, 0, W - 1))) for r, c in s]


# ----------------------------------------------------------------------------------------------
# pre-processing: PSF/Wiener, SPCNN, classical controls
# ----------------------------------------------------------------------------------------------
def degrade(img, r=2):
    import cv2
    h, w = img.shape
    lr = cv2.resize(img, (w // r, h // r), interpolation=cv2.INTER_AREA)
    return cv2.resize(lr, (w, h), interpolation=cv2.INTER_CUBIC), lr


def estimate_psf(train_imgs, lam=1e-3, size=15):
    """Eq. (1): element-wise average of F(degraded)/(F(original)+lambda*1) over training images,
    inverse-transformed, Hann-windowed (size x size) and normalised to unit sum."""
    acc = np.zeros(train_imgs[0].shape, complex)
    for im in train_imgs:
        dg, _ = degrade(im)
        acc += np.fft.fft2(dg) / (np.fft.fft2(im) + lam)
    otf = acc / len(train_imgs)
    psf = np.real(np.fft.fftshift(np.fft.ifft2(otf)))
    c0, c1 = np.array(psf.shape) // 2; h = size // 2
    psf = psf[c0 - h:c0 + h + 1, c1 - h:c1 + h + 1]
    psf = psf * np.outer(np.hanning(size), np.hanning(size))
    psf = np.clip(psf, 0, None); psf /= psf.sum()
    return psf


def wiener(img, psf, balance=1e-2):
    return np.clip(restoration.wiener(img, psf, balance, clip=False), 0, 1).astype(np.float32)


class SPCNN(nn.Module):
    """Six-layer sub-pixel CNN (Shi et al., CVPR 2016 style): five 3x3 conv (16,32,32,64,64), each followed by
    ReLU and 3x3 max-pooling with stride 1 (resolution preserved), sixth conv -> r^2=4 maps, depth-to-space x2."""
    def __init__(self, r=2):
        super().__init__()
        ch = [1, 16, 32, 32, 64, 64]
        layers = []
        for a, b in zip(ch[:-1], ch[1:]):
            layers += [nn.Conv2d(a, b, 3, padding=1), nn.ReLU(), nn.MaxPool2d(3, 1, 1)]
        self.body = nn.Sequential(*layers)
        self.last = nn.Conv2d(64, r * r, 3, padding=1)
        self.ps = nn.PixelShuffle(r)

    def forward(self, x):
        return self.ps(self.last(self.body(x)))


def train_spcnn(train_imgs, val_imgs, seed=0, epochs=40, patience=8, log=None):
    import cv2
    if os.environ.get('SMOKE') == '1':
        epochs = 1
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = SPCNN(); opt = torch.optim.Adam(net.parameters(), 1e-4)

    def patches(imgs, n=4):
        X, Y = [], []
        for im in imgs:
            _, lr = degrade(im)
            for _ in range(n):
                r, c = rng.integers(0, 256 - 64, 2) // 2 * 2
                Y.append(im[r:r + 64, c:c + 64]); X.append(lr[r // 2:r // 2 + 32, c // 2:c // 2 + 32])
        return torch.tensor(np.stack(X))[:, None], torch.tensor(np.stack(Y))[:, None]
    Xv, Yv = patches(val_imgs, 2)
    best, bad, state = 1e9, 0, None
    for ep in range(epochs):
        X, Y = patches(train_imgs)
        perm = torch.randperm(len(X)); net.train()
        for i in range(0, len(X), 16):
            idx = perm[i:i + 16]
            loss = F.mse_loss(net(X[idx]), Y[idx]); opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            vl = F.mse_loss(net(Xv), Yv).item()
        if vl < best - 1e-7:
            best, bad, state = vl, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    net.load_state_dict(state); net.eval()
    return net


def spcnn_apply(net, img):
    """Inference path (stated explicitly in the revision): the 256x256 image is fed at native resolution,
    the network outputs 512x512, which is area-resampled back to 256x256. The SPCNN therefore acts as a
    learned filter at native resolution, not as a super-resolution step."""
    import cv2
    with torch.no_grad():
        up = net(torch.tensor(img)[None, None]).numpy()[0, 0]
    return np.clip(cv2.resize(up, img.shape[::-1], interpolation=cv2.INTER_AREA), 0, 1).astype(np.float32)


def spcnn_sr(net, lr):
    with torch.no_grad():
        return np.clip(net(torch.tensor(lr)[None, None]).numpy()[0, 0], 0, 1)


def perona_malik(img, n_iter=10, kappa=0.05, gamma=0.2):
    u = img.astype(np.float32).copy()
    for _ in range(n_iter):
        dn = np.roll(u, -1, 0) - u; ds = np.roll(u, 1, 0) - u; de = np.roll(u, -1, 1) - u; dw = np.roll(u, 1, 1) - u
        g = lambda d: np.exp(-(d / kappa) ** 2)
        u = u + gamma * (g(dn) * dn + g(ds) * ds + g(de) * de + g(dw) * dw)
    return np.clip(u, 0, 1)


def available_filters():
    """Classical pre-filter controls. BM3D is skipped (with a warning) only if the bm3d package cannot be imported
    (Colab: see the troubleshooting notes); all other controls are always run."""
    ks = ['gaussian', 'bilateral', 'aniso', 'nlm', 'bm3d']
    try:
        import bm3d  # noqa: F401
    except Exception as ex:  # pragma: no cover
        print('WARNING: bm3d unavailable (%r) - the BM3D control is skipped' % (ex,), flush=True)
        ks.remove('bm3d')
    return tuple(ks)


def classical_filter(img, kind):
    if kind == 'none':
        return img
    if kind == 'gaussian':
        return filters.gaussian(img, sigma=1.0).astype(np.float32)
    if kind == 'bilateral':
        return restoration.denoise_bilateral(img, sigma_color=0.05, sigma_spatial=2).astype(np.float32)
    if kind == 'aniso':
        return perona_malik(img)
    sig = float(restoration.estimate_sigma(img))
    if kind == 'nlm':
        return restoration.denoise_nl_means(img, h=0.8 * sig, sigma=sig, patch_size=5, patch_distance=6, fast_mode=True).astype(np.float32)
    if kind == 'bm3d':
        import bm3d
        return np.clip(bm3d.bm3d(img, sigma_psd=sig), 0, 1).astype(np.float32)
    raise ValueError(kind)


# ----------------------------------------------------------------------------------------------
# superpixel region-adjacency graph
# ----------------------------------------------------------------------------------------------
def build_graph(img, scr, n_seg=400, compact=0.1, alpha=10.0, feats='enriched', use_seeds=True, mask=None):
    sp = slic(img, n_segments=n_seg, compactness=compact, channel_axis=None, start_label=0, enforce_connectivity=True)
    n = sp.max() + 1
    idx = np.arange(n)
    cnt = np.bincount(sp.ravel(), minlength=n).astype(np.float32)
    mean = ndi.mean(img, sp, idx).astype(np.float32)
    # adjacency from 4-neighbour label transitions
    a = np.concatenate([sp[:, :-1].ravel(), sp[:-1, :].ravel()]); b = np.concatenate([sp[:, 1:].ravel(), sp[1:, :].ravel()])
    m = a != b; a, b = a[m], b[m]
    e = np.unique(np.stack([np.minimum(a, b), np.maximum(a, b)], 1), axis=0)
    w = np.exp(-alpha * (mean[e[:, 0]] - mean[e[:, 1]]) ** 2).astype(np.float32)  # Eqs. (3)-(4)
    cols = [mean[:, None]]
    if feats == 'enriched':
        std = np.sqrt(np.maximum(ndi.mean(img ** 2, sp, idx) - mean ** 2, 0)).astype(np.float32)
        gm = filters.sobel(img)
        grad = ndi.mean(gm, sp, idx).astype(np.float32)
        p10 = ndi.labeled_comprehension(img, sp, idx, lambda v: np.percentile(v, 10), np.float32, 0)
        p90 = ndi.labeled_comprehension(img, sp, idx, lambda v: np.percentile(v, 90), np.float32, 0)
        rr, cc = np.mgrid[:img.shape[0], :img.shape[1]]
        cy = ndi.mean(rr, sp, idx) / img.shape[0]; cx = ndi.mean(cc, sp, idx) / img.shape[1]
        # local contrast: node mean minus mean of neighbours
        nb_sum = np.zeros(n); nb_cnt = np.zeros(n)
        np.add.at(nb_sum, e[:, 0], mean[e[:, 1]]); np.add.at(nb_cnt, e[:, 0], 1)
        np.add.at(nb_sum, e[:, 1], mean[e[:, 0]]); np.add.at(nb_cnt, e[:, 1], 1)
        contrast = mean - nb_sum / np.maximum(nb_cnt, 1)
        cols += [std[:, None], grad[:, None], p10[:, None], p90[:, None], cy[:, None], cx[:, None],
                 (cnt / cnt.mean())[:, None], contrast[:, None]]
    X = np.concatenate(cols, 1).astype(np.float32)
    seed = np.full(n, -1, np.int64)
    if use_seeds and scr is not None:
        for c in (0, 1):
            hit = sp[scr == c]
            if hit.size:
                vals, cts = np.unique(hit, return_counts=True)
                for v, ct in zip(vals, cts):
                    # majority scribble class per covered superpixel
                    other = (sp[scr == (1 - c)] == v).sum()
                    if ct >= other:
                        seed[v] = c
        sf = np.zeros((n, 3), np.float32)  # [lesion seed, background seed, is-seeded] = 3 features
        sf[seed == 1, 0] = 1; sf[seed == 0, 1] = 1; sf[seed >= 0, 2] = 1
        if feats == 'enriched':
            # graph (hop) distance from the nearest lesion / background seed, normalised
            hop = graph_hops(n, e, seed)
            sf = np.concatenate([sf, hop], 1)
        X = np.concatenate([X, sf], 1)
    y = None
    if mask is not None:
        frac = ndi.mean(mask.astype(np.float32), sp, idx)
        y = (frac >= 0.5).astype(np.int64)
    return dict(sp=sp, n=n, X=X, edges=e, w=w, seed=seed, y=y)


def graph_hops(n, e, seed, cap=20):
    adj = [[] for _ in range(n)]
    for a, b in e:
        adj[a].append(b); adj[b].append(a)
    out = np.full((n, 2), cap, np.float32)
    for k, c in enumerate((1, 0)):
        src = np.where(seed == c)[0]
        if len(src) == 0:
            continue
        d = np.full(n, cap, np.float32); d[src] = 0; q = list(src); h = 0
        while q and h < cap:
            h += 1; nq = []
            for u in q:
                for v in adj[u]:
                    if d[v] > h:
                        d[v] = h; nq.append(v)
            q = nq
        out[:, k] = d
    return out / cap


# ----------------------------------------------------------------------------------------------
# GCN models
# ----------------------------------------------------------------------------------------------
def batch_graphs(gs):
    """Dense, zero-padded batch: X (B,N,F), Lt and Am (B,N,N), valid-node mask (B,N)."""
    Bn = len(gs); N = max(g['n'] for g in gs); Fd = gs[0]['X'].shape[1]
    X = np.zeros((Bn, N, Fd), np.float32); Lt = np.zeros((Bn, N, N), np.float32); Am = np.zeros((Bn, N, N), np.float32)
    valid = np.zeros((Bn, N), bool); ys, seeds = [], []
    for k, g in enumerate(gs):
        n = g['n']; e = g['edges']; w = g['w']
        X[k, :n] = g['X']; valid[k, :n] = True
        Wm = np.zeros((n, n), np.float32); Wm[e[:, 0], e[:, 1]] = w; Wm[e[:, 1], e[:, 0]] = w
        dinv = 1 / np.sqrt(np.maximum(Wm.sum(1), 1e-6))
        # scaled Laplacian with lambda_max ~ 2:  L~ = L - I = -D^-1/2 W D^-1/2 (weighted; used by ChebNet)
        Lt[k, :n, :n] = -dinv[:, None] * Wm * dinv[None]
        # unweighted mean over self U neighbours (GraphSAGE, Eq. 8 -- ignores edge weights)
        A1 = (Wm > 0).astype(np.float32) + np.eye(n, dtype=np.float32)
        Am[k, :n, :n] = A1 / A1.sum(1, keepdims=True)
        if g['y'] is not None:
            ys.append(g['y'])
        seeds.append(g['seed'])
    vm = torch.tensor(valid)
    y = torch.tensor(np.concatenate(ys)) if ys else None
    return dict(X=torch.tensor(X), Lt=torch.tensor(Lt), Am=torch.tensor(Am), valid=vm, y=y,
                seed=torch.tensor(np.concatenate(seeds)))


class Cheb(nn.Module):
    def __init__(self, i, o, K=3):
        super().__init__(); self.K = K; self.lin = nn.Linear(i * (K + 1), o)

    def forward(self, x, Lt):
        T = [x, torch.bmm(Lt, x)]
        for _ in range(2, self.K + 1):
            T.append(2 * torch.bmm(Lt, T[-1]) - T[-2])
        return self.lin(torch.cat(T[:self.K + 1], -1))


class Sage(nn.Module):
    def __init__(self, i, o):
        super().__init__(); self.lin = nn.Linear(i, o)

    def forward(self, x, Am):
        return F.normalize(self.lin(torch.bmm(Am, x)), dim=-1)


class DualGCN(nn.Module):
    """branch = 'dual' | 'cheb' | 'sage'. Each branch has two graph-convolution layers (hidden 32 -> 16).
    Receptive field: two ChebNet layers with K=3 -> 6 hops; two GraphSAGE layers -> 2 hops.
    Fusion: concatenation + node-wise linear layer (the '1x1 convolution' of a graph). Dropout 0.5 after
    every graph-convolution layer (none after the concatenation)."""
    def __init__(self, in_dim, branch='dual', hid=32, out=16, p=0.5, K=3):
        super().__init__()
        self.branch = branch; self.p = p
        if branch in ('dual', 'cheb'):
            self.c1, self.c2 = Cheb(in_dim, hid, K), Cheb(hid, out, K)
        if branch in ('dual', 'sage'):
            self.s1, self.s2 = Sage(in_dim, hid), Sage(hid, out)
        self.head = nn.Linear(out * (2 if branch == 'dual' else 1), 2)
        self.norm = nn.LayerNorm(in_dim)

    def forward(self, B):
        x = self.norm(B['X']); outs = []
        if self.branch in ('dual', 'cheb'):
            h = F.dropout(F.relu(self.c1(x, B['Lt'])), self.p, self.training)
            h = F.dropout(F.relu(self.c2(h, B['Lt'])), self.p, self.training); outs.append(h)
        if self.branch in ('dual', 'sage'):
            h = F.dropout(F.relu(self.s1(x, B['Am'])), self.p, self.training)
            h = F.dropout(F.relu(self.s2(h, B['Am'])), self.p, self.training); outs.append(h)
        return self.head(torch.cat(outs, -1))[B['valid']]


def train_gcn(tr_graphs, va_graphs, branch='dual', seed=0, epochs=150, patience=15, bs=8, n_perm=4):
    if os.environ.get('SMOKE') == '1':
        epochs = 3
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = DualGCN(tr_graphs[0]['X'].shape[1], branch)
    opt = torch.optim.Adam(net.parameters(), 1e-3)
    ytr = np.concatenate([g['y'] for g in tr_graphs])
    pos = ytr.mean(); cw = torch.tensor([0.5 / (1 - pos), 0.5 / pos], dtype=torch.float32)  # inverse class frequency
    VB = batch_graphs(va_graphs)
    # mini-batches of 8 graphs are pre-assembled for n_perm random orderings and cycled over epochs
    perms = [rng.permutation(len(tr_graphs)) for _ in range(n_perm)]
    # dense batches are cached only for small graphs (memory); large graphs are batched on the fly (same order)
    small = max(g['n'] for g in tr_graphs) <= 600

    class _Lazy(list):
        pass
    if small:
        batches = [[batch_graphs([tr_graphs[j] for j in p[i:i + bs]]) for i in range(0, len(p), bs)] for p in perms]
    else:
        batches = [[(lambda idx: (lambda: batch_graphs([tr_graphs[j] for j in idx])))(p[i:i + bs]) for i in range(0, len(p), bs)] for p in perms]
    best, bad, state = 1e9, 0, None
    for ep in range(epochs):
        net.train(); bl = batches[ep % n_perm]
        for k in rng.permutation(len(bl)):
            B = bl[k] if small else bl[k]()
            loss = F.cross_entropy(net(B), B['y'], weight=cw)
            opt.zero_grad(); loss.backward(); opt.step()
        net.eval()
        with torch.no_grad():
            vl = F.cross_entropy(net(VB), VB['y'], weight=cw).item()
        if vl < best - 1e-5:
            best, bad, state = vl, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break
    net.load_state_dict(state); net.eval()
    net.n_epochs = ep + 1; net.pos = float(pos)
    return net


def prior_correct(p, pos):
    """Undo the shift introduced by inverse-class-frequency loss weights (w1 = 0.5/pi, w0 = 0.5/(1-pi)):
    a network trained with weights w_c estimates q(c|x) proportional to w_c p(c|x), so p(c|x) is proportional to q(c|x)/w_c."""
    w1, w0 = 0.5 / pos, 0.5 / (1 - pos)
    a, b = p / w1, (1 - p) / w0
    return a / (a + b + 1e-12)


def gcn_predict(net, g, keep_seeds=True, correct=False):
    B = batch_graphs([g])
    with torch.no_grad():
        p = F.softmax(net(B), 1)[:, 1].numpy()
    if correct:
        p = prior_correct(p, net.pos)
    if keep_seeds:
        p = np.where(g['seed'] == 1, 1.0, np.where(g['seed'] == 0, 0.0, p))
    return p[g['sp']].astype(np.float32)  # pixel-level posterior map


# ----------------------------------------------------------------------------------------------
# training-free seeded baselines
# ----------------------------------------------------------------------------------------------
def rw_segment(img, scr, beta=130):
    if not (scr == 1).any():
        return np.zeros(img.shape, np.float32)
    lab = np.zeros(img.shape, np.int32); lab[scr == 0] = 1; lab[scr == 1] = 2
    if not (scr == 0).any():
        return np.ones(img.shape, np.float32)
    out = random_walker(img, lab, beta=beta, mode='cg_j', return_full_prob=True)
    return out[1].astype(np.float32)


def label_propagation(g, alpha=0.99, iters=50, cmn=False):
    """Zhou et al. (2004) on the same superpixel graph and edge weights as the GCN."""
    n = g['n']; e = g['edges']; w = g['w']
    Y = np.zeros((n, 2)); Y[g['seed'] == 0, 0] = 1; Y[g['seed'] == 1, 1] = 1
    if Y[:, 1].sum() == 0:
        return np.zeros(g['sp'].shape, np.float32)
    Wm = np.zeros((n, n)); Wm[e[:, 0], e[:, 1]] = w; Wm[e[:, 1], e[:, 0]] = w
    d = Wm.sum(1); Dm = 1 / np.sqrt(np.maximum(d, 1e-9)); S = Dm[:, None] * Wm * Dm[None]
    Fm = Y.copy()
    for _ in range(iters):
        Fm = alpha * S @ Fm + (1 - alpha) * Y
    if cmn:
        Fm = Fm / np.maximum(Fm.sum(0, keepdims=True), 1e-12)  # class-mass normalisation (Zhu et al. 2003)
    p = Fm[:, 1] / np.maximum(Fm.sum(1), 1e-12)
    return p[g['sp']].astype(np.float32)


# ----------------------------------------------------------------------------------------------
# pixel-level refinement: convolutional CRF (ConvCRF, Teichmann & Cipolla 2019) mean-field
# ----------------------------------------------------------------------------------------------
def conv_crf(prob, img, k=11, theta_a=3.0, theta_b=0.05, theta_g=1.5, w_app=4.0, w_sm=2.0, iters=5, scr=None):
    """Mean-field inference of a locally-connected (k x k) CRF with an appearance (bilateral) kernel
    and a smoothness kernel, initialised by the GCN posterior; scribbled pixels are clamped."""
    P = torch.tensor(np.stack([1 - prob, prob]))[None].float().clamp(1e-5, 1 - 1e-5)
    U = -torch.log(P)
    I = torch.tensor(img)[None, None].float()
    pad = k // 2
    Iu = F.unfold(I, k, padding=pad)  # 1 x k*k x HW
    dI = (Iu - I.reshape(1, 1, -1)) ** 2
    yy, xx = torch.meshgrid(torch.arange(k) - pad, torch.arange(k) - pad, indexing='ij')
    d2 = (yy ** 2 + xx ** 2).float().reshape(1, -1, 1)
    Ka = w_app * torch.exp(-d2 / (2 * theta_a ** 2) - dI / (2 * theta_b ** 2))
    Ks = w_sm * torch.exp(-d2 / (2 * theta_g ** 2)).expand_as(Ka)
    K = Ka + Ks; K[:, k * k // 2] = 0
    Q = P.clone(); H, W = prob.shape
    for _ in range(iters):
        msg = torch.cat([(F.unfold(Q[:, ch:ch + 1], k, padding=pad) * K).sum(1).reshape(1, 1, H, W) for ch in range(2)], 1)
        pair = msg.flip(1)  # Potts compatibility: penalise the other label
        Q = F.softmax(-U - pair, 1)
    q = Q[0, 1].numpy()
    if scr is not None:
        q = np.where(scr == 1, 1.0, np.where(scr == 0, 0.0, q))
    return q.astype(np.float32)


# ----------------------------------------------------------------------------------------------
# U-Net family (2D U-Net of Ronneberger et al., nnU-Net-style training ingredients)
# ----------------------------------------------------------------------------------------------
class Block(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.b = nn.Sequential(nn.Conv2d(i, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o, affine=True), nn.LeakyReLU(0.01),
                               nn.Conv2d(o, o, 3, padding=1, bias=False), nn.InstanceNorm2d(o, affine=True), nn.LeakyReLU(0.01))

    def forward(self, x):
        return self.b(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, base=16, depth=4):
        super().__init__()
        ch = [base * 2 ** i for i in range(depth + 1)]
        self.enc = nn.ModuleList([Block(in_ch, ch[0])] + [Block(ch[i], ch[i + 1]) for i in range(depth)])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i + 1], ch[i], 2, 2) for i in range(depth)][::-1])
        self.dec = nn.ModuleList([Block(ch[i] * 2, ch[i]) for i in range(depth)][::-1])
        self.out = nn.Conv2d(ch[0], 2, 1)
        self.ds = nn.Conv2d(ch[1], 2, 1)  # deep supervision head

    def forward(self, x):
        skips = []
        for i, b in enumerate(self.enc):
            x = b(x if i == 0 else F.max_pool2d(x, 2)); skips.append(x)
        x = skips.pop(); aux = None
        for j, (u, d) in enumerate(zip(self.up, self.dec)):
            x = d(torch.cat([u(x), skips.pop()], 1))
            if j == len(self.dec) - 2:
                aux = self.ds(x)
        return (self.out(x), aux) if self.training else self.out(x)


def soft_dice_loss(logits, y):
    p = F.softmax(logits, 1)[:, 1]; yf = y.float()
    inter = (p * yf).sum((1, 2)); den = p.sum((1, 2)) + yf.sum((1, 2))
    return (1 - (2 * inter + 1) / (den + 1)).mean()


def augment(stack, rng):
    """stack: C x H x W (last channel = label). Rotation +-10 deg, shift/zoom 5 %, horizontal flip only
    (vertical flips removed in the revision as anatomically implausible for sagittal sections)."""
    import cv2
    C, H, W = stack.shape
    ang = rng.uniform(-10, 10); sc = rng.uniform(0.95, 1.05); tx, ty = rng.uniform(-0.05, 0.05, 2) * W
    M = cv2.getRotationMatrix2D((W / 2, H / 2), ang, sc); M[:, 2] += (tx, ty)
    out = np.stack([cv2.warpAffine(stack[c], M, (W, H), flags=cv2.INTER_NEAREST if c == C - 1 else cv2.INTER_LINEAR,
                                   borderMode=cv2.BORDER_REFLECT) for c in range(C)])
    if rng.random() < 0.5:
        out = out[:, :, ::-1].copy()
    return out


def train_unet(inputs, labels, val_inputs, val_labels, in_ch, seed=0, epochs=60, patience=12, patch=128, bs=8,
               aug_fn=None, steps_per_epoch=None, min_epochs=30):
    """inputs: list of (C,H,W) arrays. Patch-based training (nnU-Net style), CE + soft Dice, deep supervision,
    poly LR, early stopping on validation Dice (full 256 x 256 images)."""
    if os.environ.get('SMOKE') == '1':
        epochs, steps_per_epoch = 1, 2
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = UNet(in_ch); opt = torch.optim.SGD(net.parameters(), 0.01, momentum=0.99, nesterov=True, weight_decay=3e-5)
    cw = torch.tensor([1.0, 1.0])  # nnU-Net default: unweighted CE + soft Dice (class weights biased the U-Net to false positives)
    n = len(inputs); spe = steps_per_epoch or max(1, n // bs)
    best, bad, state = -1, 0, None
    for ep in range(epochs):
        lr = 0.01 * (1 - ep / epochs) ** 0.9
        for g_ in opt.param_groups:
            g_['lr'] = lr
        net.train()
        for _ in range(spe):
            Xb, Yb = [], []
            for j in rng.integers(0, n, bs):
                st = np.concatenate([inputs[j], labels[j][None].astype(np.float32)], 0)
                st = augment(st, rng)
                H, W = st.shape[1:]
                # 1/3 of patches forced to contain foreground (nnU-Net oversampling)
                if rng.random() < 0.33 and st[-1].any():
                    fg = np.argwhere(st[-1] > 0.5); r, c = fg[rng.integers(len(fg))]
                    r0 = int(np.clip(r - patch // 2 + rng.integers(-patch // 4, patch // 4 + 1), 0, H - patch))
                    c0 = int(np.clip(c - patch // 2 + rng.integers(-patch // 4, patch // 4 + 1), 0, W - patch))
                else:
                    r0, c0 = rng.integers(0, H - patch + 1), rng.integers(0, W - patch + 1)
                st = st[:, r0:r0 + patch, c0:c0 + patch]
                Xb.append(st[:-1]); Yb.append(st[-1] > 0.5)
            X = torch.tensor(np.stack(Xb)).float(); Y = torch.tensor(np.stack(Yb)).long()
            out, aux = net(X)
            Ya = F.max_pool2d(Y[:, None].float(), 2)[:, 0].long()
            loss = F.cross_entropy(out, Y, weight=cw) + soft_dice_loss(out, Y) + \
                0.5 * (F.cross_entropy(aux, Ya, weight=cw) + soft_dice_loss(aux, Ya))
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(net.parameters(), 12); opt.step()
        net.eval()
        vd = np.mean([dice(unet_predict(net, x) > 0.5, y) for x, y in zip(val_inputs, val_labels) if y.any()])
        if vd > best + 1e-4:
            best, bad, state = vd, 0, {k: v.clone() for k, v in net.state_dict().items()}
        else:
            bad += 1
            if bad >= patience and ep + 1 >= min_epochs:
                break
    net.load_state_dict(state); net.eval()
    net.n_epochs = ep + 1; net.best_val = best
    return net


def unet_predict(net, x):
    with torch.no_grad():
        return F.softmax(net(torch.tensor(x)[None].float()), 1)[0, 1].numpy()


def geodesic_maps(img, scr, lam=10.0):
    """DeepIGeoS-style geodesic distance of every pixel to the lesion / background scribbles."""
    gm = filters.sobel(img)
    cost = 1.0 + lam * gm / (gm.max() + 1e-6)
    out = []
    for c in (1, 0):
        seeds = np.argwhere(scr == c)
        if len(seeds) == 0:
            out.append(np.ones(img.shape, np.float32)); continue
        m = skgraph.MCP_Geometric(cost)
        d, _ = m.find_costs([tuple(s) for s in seeds])
        out.append(np.clip(d / 256.0, 0, 1).astype(np.float32))
    return np.stack(out)


# ----------------------------------------------------------------------------------------------
# metrics
# ----------------------------------------------------------------------------------------------
def dice(p, g):
    p = p.astype(bool); g = g.astype(bool)
    s = p.sum() + g.sum()
    return 1.0 if s == 0 else 2 * (p & g).sum() / s


def seg_metrics(pred, gt):
    p = pred.astype(bool); g = gt.astype(bool)
    tp = (p & g).sum(); fp = (p & ~g).sum(); fn = (~p & g).sum(); tn = (~p & ~g).sum()
    d = 2 * tp / max(2 * tp + fp + fn, 1); iou = tp / max(tp + fp + fn, 1)
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0; rec = tp / max(tp + fn, 1)
    hd95, assd = surface_distances(p, g)
    return dict(dice=d, iou=iou, precision=prec, recall=rec, accuracy=(tp + tn) / p.size, hd95=hd95, assd=assd,
                area_pred=p.sum() * PIX_MM ** 2, area_gt=g.sum() * PIX_MM ** 2, empty=int(p.sum() == 0))


def surface_distances(p, g):
    if p.sum() == 0 or g.sum() == 0:
        return np.nan, np.nan
    bp = p & ~ndi.binary_erosion(p); bg = g & ~ndi.binary_erosion(g)
    dg = ndi.distance_transform_edt(~bg) * PIX_MM; dp = ndi.distance_transform_edt(~bp) * PIX_MM
    d1 = dg[bp]; d2 = dp[bg]
    allp = np.concatenate([d1, d2])
    return float(np.percentile(allp, 95)), float(allp.mean())


def boxes_from_mask(m, min_area=20):
    lab = cclabel(m.astype(np.uint8), connectivity=2)
    out = []
    for r in regionprops(lab):
        if r.area >= min_area:
            r0, c0, r1, c1 = r.bbox; out.append([c0, r0, c1, r1])
    return np.array(out, np.float32).reshape(-1, 4)


def boxes_from_prob(prob, thr=0.5, min_area=20):
    """Connected regions of the segmentation become boxes; score = mean posterior in the region."""
    lab = cclabel((prob > thr).astype(np.uint8), connectivity=2)
    bx, sc = [], []
    for r in regionprops(lab):
        if r.area >= min_area:
            r0, c0, r1, c1 = r.bbox; bx.append([c0, r0, c1, r1]); sc.append(float(prob[lab == r.label].mean()))
    return np.array(bx, np.float32).reshape(-1, 4), np.array(sc, np.float32)

Writing /content/pal_code/pal/core.py


`httpfile.py` — seekable HTTP(S) file with Range requests (new; replaces the authors' private helper)

In [ ]:
%%writefile /content/pal_code/httpfile.py
"""Seekable, read-only file object over HTTP(S) using Range requests.
Lets Python's zipfile read single members of the 42.6 GB DOLCHID.zip without downloading the whole archive.
Handles redirects (figshare -> signed S3 URL), expired signed URLs and transient network errors."""
import io, time, collections
import requests


class HttpFile(io.RawIOBase):
    def __init__(self, url, block=4 * 1024 * 1024, cache_blocks=32, retries=6, timeout=120):
        self.url0 = url; self.block = block; self.retries = retries; self.timeout = timeout
        self.sess = requests.Session()
        self.sess.headers['User-Agent'] = 'pal-colab-reader/1.0'
        self._cache = collections.OrderedDict(); self._max = cache_blocks
        self.pos = 0
        self._resolve()

    # ------------------------------------------------------------------ helpers
    def _resolve(self):
        """Follow redirects once and learn the total size from a 1-byte range request."""
        last = None
        for k in range(self.retries):
            try:
                r = self.sess.get(self.url0, headers={'Range': 'bytes=0-0'}, stream=True, allow_redirects=True, timeout=self.timeout)
                if r.status_code == 206 and 'Content-Range' in r.headers:
                    self.url = r.url
                    self.size = int(r.headers['Content-Range'].split('/')[-1]); r.close(); return
                if r.status_code == 200:
                    r.close()
                    raise IOError('Server ignored the Range header (HTTP 200). Use the direct file download URL '
                                  '(https://figshare.com/ndownloader/files/<FILE_ID>), not the article page.')
                last = IOError(f'HTTP {r.status_code} while resolving {self.url0}')
                r.close()
            except (requests.RequestException, IOError) as ex:
                last = ex
                if 'ignored the Range' in str(ex):
                    raise
            time.sleep(2 ** k)
        raise last

    def _fetch(self, start, end):
        last = None
        for k in range(self.retries):
            try:
                r = self.sess.get(self.url, headers={'Range': f'bytes={start}-{end}'}, timeout=self.timeout)
                if r.status_code == 206:
                    return r.content
                if r.status_code in (400, 403, 410):      # signed URL expired -> resolve again
                    self._resolve()
                last = IOError(f'HTTP {r.status_code} for bytes {start}-{end}')
            except requests.RequestException as ex:
                last = ex
            time.sleep(2 ** k)
        raise last

    def _block(self, b):
        if b in self._cache:
            self._cache.move_to_end(b); return self._cache[b]
        s = b * self.block; e = min(s + self.block, self.size) - 1
        data = self._fetch(s, e)
        self._cache[b] = data
        if len(self._cache) > self._max:
            self._cache.popitem(last=False)
        return data

    # ------------------------------------------------------------------ file API
    def readable(self): return True
    def seekable(self): return True
    def tell(self): return self.pos

    def seek(self, off, whence=io.SEEK_SET):
        if whence == io.SEEK_SET: self.pos = off
        elif whence == io.SEEK_CUR: self.pos += off
        elif whence == io.SEEK_END: self.pos = self.size + off
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = max(0, min(n, self.size - self.pos))
        if n == 0:
            return b''
        if n > 4 * self.block:                            # large member: one direct streaming request
            data = self._fetch(self.pos, self.pos + n - 1)
        else:
            out = []; p = self.pos; end = self.pos + n
            while p < end:
                b = p // self.block; blk = self._block(b); off = p - b * self.block
                take = blk[off:off + (end - p)]; out.append(take); p += len(take)
            data = b''.join(out)
        self.pos += len(data)
        return data

    def readinto(self, buf):
        d = self.read(len(buf)); buf[:len(d)] = d; return len(d)

Writing /content/pal_code/httpfile.py


`extract_sections.py` — DOLCHID section extraction (data access generalised for Colab)

In [ ]:
%%writefile /content/pal_code/extract_sections.py
"""Download DOLCHID volumes member-by-member (HTTP range requests) and extract 2D sagittal sections.
Colab version: the processing (windowing, section positions, ROI, seeded offsets) is identical to the authors'
script; only the data access is generalised. Choose ONE data source with environment variables:

  PAL_DOLCHID_URL  direct download URL of DOLCHID.zip (https://figshare.com/ndownloader/files/<FILE_ID>) - streamed
  PAL_DOLCHID_ZIP  path to a local / Google-Drive copy of DOLCHID.zip
  PAL_DOLCHID_DIR  folder that contains the extracted *_CBCT_Image.nii.gz and *_CBCT_Label.nii.gz files
  PAL_LABEL_DIR    (optional) separate folder with the *_CBCT_Label.nii.gz files (the authors' 'labels_all/')

usage:  python extract_sections.py RC RC_1 RC_2 ... RC_54
        python extract_sections.py OTHER DC_1 ... DC_6 KCOT_1 ... KCOT_6 AME_1 ... AME_6
"""
import sys, os, zipfile, tempfile, glob
import numpy as np, nibabel as nib

PAL_WORK = os.environ.get('PAL_WORK', '/content/pal_work')
OUT = os.environ.get('PAL_DATA', PAL_WORK + '/data')
LO, HI = -1000.0, 3000.0


def win(x):
    return np.clip((x.astype(np.float32) - LO) / (HI - LO), 0, 1)


def sag(vol, x):
    # vol axes (L,P,S) -> sagittal image rows = superior->inferior, cols = anterior->posterior
    return np.flipud(vol[x].T)


def crop(im, r0, c0, S=256):
    out = np.zeros((S, S), im.dtype)
    H, W = im.shape
    rs, cs = max(r0, 0), max(c0, 0); re, ce = min(r0 + S, H), min(c0 + S, W)
    out[rs - r0:re - r0, cs - c0:ce - c0] = im[rs:re, cs:ce]
    return out


# ---------------------------------------------------------------------------------------------- data access
class Source:
    """Finds '<pid>_CBCT_Image.nii.gz' / '<pid>_CBCT_Label.nii.gz' wherever they are in the zip or folder."""
    def __init__(self):
        url = os.environ.get('PAL_DOLCHID_URL', '').strip(); zp = os.environ.get('PAL_DOLCHID_ZIP', '').strip()
        dr = os.environ.get('PAL_DOLCHID_DIR', '').strip(); self.label_dir = os.environ.get('PAL_LABEL_DIR', '').strip()
        self.z = None; self.files = {}
        if url:
            from httpfile import HttpFile
            self.z = zipfile.ZipFile(HttpFile(url))
        elif zp:
            self.z = zipfile.ZipFile(zp)
        elif dr:
            scan = [d for d in (os.environ.get('PAL_IMAGE_DIR', '').strip(), os.environ.get('PAL_LABEL_DIR', '').strip()) if d] or [dr]
            for d in scan:
                for p in glob.glob(os.path.join(d, '**', '*.nii*'), recursive=True):
                    self.files[os.path.basename(p)] = p
        else:
            raise SystemExit('Set PAL_DOLCHID_URL, PAL_DOLCHID_ZIP or PAL_DOLCHID_DIR (see the notebook, Step 4).')
        self.members = {}
        if self.z is not None:
            for n in self.z.namelist():
                self.members.setdefault(os.path.basename(n), n)
        if self.label_dir:
            for p in glob.glob(os.path.join(self.label_dir, '**', '*.nii*'), recursive=True):
                self.files.setdefault(os.path.basename(p), p)

    def _find(self, fname):
        alts = [fname, fname.replace('.nii.gz', '.nii')]
        for a in alts:
            if a in self.files:
                return ('file', self.files[a])
        for a in alts:
            if a in self.members:
                return ('zip', self.members[a])
        # case-insensitive fallback
        low = fname.lower().replace('.nii.gz', '')
        for k, v in list(self.files.items()):
            if k.lower().startswith(low):
                return ('file', v)
        for k, v in self.members.items():
            if k.lower().startswith(low):
                return ('zip', v)
        raise FileNotFoundError(f'{fname} not found in the DOLCHID source. Run the "list archive" cell to see the real member names.')

    def load(self, fname):
        kind, ref = self._find(fname)
        if kind == 'file':
            return np.asarray(nib.load(ref).dataobj)
        with self.z.open(ref) as f:
            raw = f.read()
        suf = '.nii.gz' if ref.endswith('.gz') else '.nii'
        tmp = tempfile.NamedTemporaryFile(suffix=suf, delete=False, dir=tempfile.gettempdir())
        tmp.write(raw); tmp.close()
        try:
            arr = np.asarray(nib.load(tmp.name).dataobj)
        finally:
            os.unlink(tmp.name)
        return arr


def main(ids, kind):
    os.makedirs(OUT, exist_ok=True)
    src = Source()
    for pid in ids:
        fn = f'{OUT}/{pid}.npz'
        if os.path.exists(fn):
            print(pid, 'exists - skipped', flush=True)
            continue
        lab = src.load(f'{pid}_CBCT_Label.nii.gz') > 0
        vol = src.load(f'{pid}_CBCT_Image.nii.gz')
        assert vol.shape == lab.shape, (vol.shape, lab.shape)
        xs = np.where(lab.any((1, 2)))[0]; xmin, xmax = xs.min(), xs.max()
        X = vol.shape[0]
        side = 1 if (X - 1 - xmax) >= xmin else -1
        xneg = xmax + 30 if side == 1 else xmin - 30
        xneg = int(np.clip(xneg, 0, X - 1))
        rng = np.random.default_rng(int(''.join(c for c in pid if c.isdigit())) + 1000 * ['RC', 'DC', 'KCOT', 'AME'].index(pid.split('_')[0]))
        lm = np.stack([sag(lab, x) for x in xs])
        rr, cc = np.where(lm.any(0)); rc, ccen = rr.mean(), cc.mean()
        r0 = int(round(rc - 128 + rng.integers(-64, 65))); c0 = int(round(ccen - 128 + rng.integers(-64, 65)))
        r0 = int(np.clip(r0, rr.max() - 255, rr.min())) if rr.max() - rr.min() < 256 else int(rr.min())
        c0 = int(np.clip(c0, cc.max() - 255, cc.min())) if cc.max() - cc.min() < 256 else int(cc.min())
        H, W = lm.shape[1:]
        r0 = int(np.clip(r0, 0, H - 256)); c0 = int(np.clip(c0, 0, W - 256))
        rec = dict(pid=pid, kind=kind, r0=r0, c0=c0, xmin=xmin, xmax=xmax, xneg=xneg, shape=vol.shape)
        tmp_fn = f'{OUT}/.tmp_{pid}.npz'   # hidden name: never matched by the RC_*.npz / [DKA]*_*.npz globs
        if kind == 'RC':
            secx = [int(round(xmin + p * (xmax - xmin))) for p in (0.25, 0.5, 0.75)] + [xneg]
            imgs, msks, fulls, fullm = [], [], [], []
            for x in secx:
                si, sm = win(sag(vol, x)), sag(lab, x).astype(np.uint8)
                imgs.append(crop(si, r0, c0)); msks.append(crop(sm, r0, c0))
                fulls.append(si.astype(np.float16)); fullm.append(sm)
            if msks[3].any():
                print('WARN neg has lesion', pid)
            ext_x = list(range(xmin, xmax + 1, 2))
            ext_i = np.stack([crop(win(sag(vol, x)), r0, c0) for x in ext_x]).astype(np.float16)
            ext_m = np.stack([crop(sag(lab, x).astype(np.uint8), r0, c0) for x in ext_x])
            np.savez_compressed(tmp_fn, img=np.stack(imgs).astype(np.float32), msk=np.stack(msks), secx=np.array(secx),
                                full_img=np.stack(fulls), full_msk=np.stack(fullm), ext_x=np.array(ext_x),
                                ext_img=ext_i, ext_msk=ext_m, meta=np.array([str(rec)]))
        else:
            si, sm = win(sag(vol, xneg)), sag(lab, xneg).astype(np.uint8)
            np.savez_compressed(tmp_fn, img=crop(si, r0, c0)[None].astype(np.float32), msk=crop(sm, r0, c0)[None],
                                meta=np.array([str(rec)]))
        os.replace(tmp_fn, fn)   # atomic: an interrupted run never leaves a half-written <pid>.npz
        print(pid, 'x', xmin, xmax, 'neg', xneg, 'win', r0, c0, 'vol range', float(vol.min()), float(np.percentile(vol, 99.9)), flush=True)
        del vol, lab


if __name__ == '__main__':
    kind = sys.argv[1]; ids = sys.argv[2:]
    main(ids, kind)

Writing /content/pal_code/extract_sections.py


`run_seg.py` — segmentation experiments for one repetition x fold

In [ ]:
%%writefile /content/pal_code/run_seg.py
"""Segmentation experiments for one (repeat, fold) unit.
usage: python run_seg.py REP FOLD [--extras]
Writes per-image rows to results/seg_r{REP}_f{FOLD}.csv (and extras for REP 0)."""
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import sys, os, time, json, pickle
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np, pandas as pd, torch
from pal.core import *

torch.set_num_threads(int(os.environ.get('NTH', '1')))
REP, FOLD = int(sys.argv[1]), int(sys.argv[2])
EXTRAS = '--extras' in sys.argv
ONLY = [a.split('=')[1] for a in sys.argv if a.startswith('--only=')]
RES = os.environ.get('RESDIR', PAL_WORK + '/results'); CACHE = os.environ.get('CACHEDIR', PAL_WORK + '/cache')
SMOKE = os.environ.get('SMOKE') == '1'
TAG = ('_' + '_'.join(ONLY)) if ONLY else ''
import zlib
os.makedirs(RES, exist_ok=True); os.makedirs(CACHE, exist_ok=True); os.makedirs(f'{RES}/pred', exist_ok=True)
LOG = open(f'{RES}/log_r{REP}_f{FOLD}{TAG}.txt', 'a')


def log(*a):
    s = time.strftime('%H:%M:%S ') + ' '.join(map(str, a)); print(s, flush=True); LOG.write(s + '\n'); LOG.flush()


ds = load_dataset()
N_IMG = len(ds['img'])
fold_of = make_folds(ds, 4, seed=REP)
pid = ds['pid']
test_idx = np.where([fold_of[p] == FOLD for p in pid])[0]
if SMOKE:
    test_idx = test_idx[:6]
train_p = sorted({p for p in pid if fold_of[p] != FOLD})
tr_p, va_p = split_train_val(train_p, seed=REP * 10 + FOLD)
tr_idx = np.where(np.isin(pid, tr_p))[0]; va_idx = np.where(np.isin(pid, va_p))[0]
log('rep', REP, 'fold', FOLD, 'train', len(tr_idx), 'val', len(va_idx), 'test', len(test_idx))

# reference lesion location for lesion-free sections (operator looks at the same region)
ref_mask = {}
for i in range(N_IMG):
    if ds['sec'][i] == 1:
        ref_mask[pid[i]] = ds['msk'][i]


def scribbles(i, mode='default', rep=REP, salt=0):
    rng = np.random.default_rng(1_000_003 * rep + 7919 * i + 104729 * salt + zlib.crc32(mode.encode()) % 9973)
    return simulate_scribbles(ds['msk'][i], rng, mode=mode, ref_mask=ref_mask[pid[i]])


# fixed augmented copy of each training/validation image (one copy, as in the manuscript)
def aug_copy(i):
    rng = np.random.default_rng(424242 + 31 * i + REP)
    st = augment(np.stack([ds['img'][i], ds['msk'][i].astype(np.float32)]), rng)
    return st[0].astype(np.float32), (st[1] > 0.5).astype(np.uint8)


AUG = {i: aug_copy(i) for i in np.concatenate([tr_idx, va_idx])}


def aug_scr(i):
    rng = np.random.default_rng(99991 * i + REP)
    m = AUG[i][1]
    return simulate_scribbles(m, rng, 'default', ref_mask=ref_mask[pid[i]] if not m.any() else None)


# ---------------------------------------------------------------------------------- pre-processing
t0 = time.time()
PCF = f'{CACHE}/pre_r{REP}_f{FOLD}.npz'
if os.path.exists(PCF) and not SMOKE:
    z = np.load(PCF, allow_pickle=True)
    psf, psf_img, enh = z['psf'], z['psf_img'], z['enh']
    ak = z['aug_keys']; psf_aug = dict(zip(ak.tolist(), z['psf_aug'])); enh_aug = dict(zip(ak.tolist(), z['enh_aug']))
    spc = SPCNN(); spc.load_state_dict(torch.load(f'{CACHE}/spcnn_r{REP}_f{FOLD}.pt')); spc.eval()
else:
    psf = estimate_psf([ds['img'][i] for i in tr_idx])
    psf_img = np.stack([wiener(im, psf) for im in ds['img']])
    psf_aug = {i: wiener(AUG[i][0], psf) for i in AUG}
    spc = train_spcnn([psf_img[i] for i in tr_idx], [psf_img[i] for i in va_idx], seed=REP * 10 + FOLD)
    enh = np.stack([spcnn_apply(spc, im) for im in psf_img])
    enh_aug = {i: spcnn_apply(spc, psf_aug[i]) for i in AUG}
    if not SMOKE:
        ak = np.array(sorted(AUG))
        np.savez(PCF, psf=psf, psf_img=psf_img, enh=enh, aug_keys=ak, psf_aug=np.stack([psf_aug[i] for i in ak]),
                 enh_aug=np.stack([enh_aug[i] for i in ak]))
        torch.save(spc.state_dict(), f'{CACHE}/spcnn_r{REP}_f{FOLD}.pt')
log('pre-processing done', round(time.time() - t0, 1), 's')
if REP == 0:
    torch.save(spc.state_dict(), f'{CACHE}/spcnn_f{FOLD}.pt'); np.save(f'{CACHE}/psf_f{FOLD}.npy', psf)
    np.save(f'{CACHE}/enh_f{FOLD}.npy', enh[test_idx].astype(np.float16))

rows = []


def record(method, i, prob, scen='default', extra=None, thr=0.5):
    pred = prob > thr
    r = dict(rep=REP, fold=FOLD, idx=int(i), pid=pid[i], sec=int(ds['sec'][i]), lesion=bool(ds['lesion'][i]),
             method=method, scenario=scen)
    if ds['lesion'][i]:
        r.update(seg_metrics(pred, ds['msk'][i]))
    else:
        r.update(dict(fp_pixels=int(pred.sum()), fp_area=float(pred.sum() * PIX_MM ** 2), fp_any=int(pred.sum() > 0)))
    if extra:
        r.update(extra)
    rows.append(r)


def flush():
    pd.DataFrame(rows).to_csv(f'{RES}/seg_r{REP}_f{FOLD}{TAG}.csv', index=False)


PRED = {}  # rep-0 probability maps for figures / detection


def keep(method, i, prob, scen='default'):
    if REP == 0 and scen == 'default':
        PRED.setdefault(method, {})[int(i)] = prob.astype(np.float16)


def graphs_for(imgs_all, imgs_aug, idxs, with_aug, n_seg=400, compact=0.1, feats='enriched', seeds=True, scen='default'):
    gs = []
    for i in idxs:
        s = scribbles(i, scen) if seeds else None
        gs.append(build_graph(imgs_all[i], s, n_seg, compact, feats=feats, use_seeds=seeds, mask=ds['msk'][i]))
        if with_aug:
            gs.append(build_graph(imgs_aug[i], aug_scr(i) if seeds else None, n_seg, compact, feats=feats,
                                  use_seeds=seeds, mask=AUG[i][1]))
    return gs


def run_gcn(name, imgs_all, imgs_aug, branch='dual', feats='enriched', seeds=True, crf=False, n_seg=400, compact=0.1,
            scenarios=('default',), save=False, crf_also=None):
    t = time.time()
    tr = graphs_for(imgs_all, imgs_aug, tr_idx, True, n_seg, compact, feats, seeds)
    va = graphs_for(imgs_all, imgs_aug, va_idx, False, n_seg, compact, feats, seeds)
    net = train_gcn(tr, va, branch, seed=REP * 100 + FOLD)
    nodes = np.mean([g['n'] for g in tr])
    for scen in scenarios:
        for i in test_idx:
            s = scribbles(i, scen) if seeds else None
            g = build_graph(imgs_all[i], s, n_seg, compact, feats=feats, use_seeds=seeds)
            p = gcn_predict(net, g, keep_seeds=seeds)
            record(name, i, p, scen, dict(nodes=g['n']))
            if save:
                keep(name, i, p, scen)
            if crf or crf_also:
                # prior-corrected posterior (Section 2.3) -> ConvCRF  = reported configuration
                pc = gcn_predict(net, g, keep_seeds=seeds, correct=True)
                record(name + ' + PC', i, pc, scen)
                q = conv_crf(pc, imgs_all[i], scr=s)
                record(crf_also or name, i, q, scen)
                if save:
                    keep(crf_also or name, i, q, scen); keep(name + ' + PC', i, pc, scen)
                if scen == 'default':
                    q2 = conv_crf(p, imgs_all[i], scr=s)  # CRF on the uncorrected (weighted) posterior, for the ablation
                    record(name + ' + CRF (no PC)', i, q2, scen)
    log(name, 'epochs', net.n_epochs, 'nodes', round(nodes, 1), 'time', round(time.time() - t, 1))
    flush()
    return net



def run_baselines():
    """Training-free seeded baselines; their one free parameter is selected on the validation patients (Dice)."""
    t = time.time()
    vl = [i for i in va_idx if ds['lesion'][i]]
    best_b, best_bd = 130, -1
    for b in (30, 130, 500):
        v = np.mean([dice(rw_segment(enh[i], scribbles(i), beta=b) > 0.5, ds['msk'][i]) for i in vl])
        if v > best_bd:
            best_b, best_bd = b, v
    vg = {i: build_graph(enh[i], scribbles(i), 400, 0.1, feats='basic') for i in vl}
    best_l, best_ld = (0.99, False), -1
    for a_ in (0.5, 0.9, 0.99):
        for cm in (False, True):
            v = np.mean([dice(label_propagation(vg[i], a_, cmn=cm) > 0.5, ds['msk'][i]) for i in vl])
            if v > best_ld:
                best_l, best_ld = (a_, cm), v
    log('RW beta', best_b, 'LP alpha/cmn', best_l)
    for scen in SCEN:
        for i in test_idx:
            s = scribbles(i, scen)
            p = rw_segment(enh[i], s, beta=best_b); record('Random walker', i, p, scen, dict(param=best_b)); keep('Random walker', i, p, scen)
            g = build_graph(enh[i], s, 400, 0.1, feats='basic')
            p = label_propagation(g, best_l[0], cmn=best_l[1]); record('Label propagation', i, p, scen, dict(param=str(best_l))); keep('Label propagation', i, p, scen)
    log('RW/LP', round(time.time() - t, 1)); flush()

ROB = ('default', 'density05', 'density30', 'boundary', 'mislabel10', 'mislabel20', 'click', 'missing', 'opA', 'opB', 'opC')
SCEN = ROB if REP == 0 else ('default',)

BASE = 'base' in ONLY
FOVM = 'fov' in ONLY
if not ONLY or 'main' in ONLY or BASE or FOVM:
    # ---------------------------------------------------------------- graph-based methods (main comparison)
    # proposed: PSF+SPCNN -> SLIC(400) -> enriched node features + seeds -> dual GCN -> ConvCRF refinement
    if BASE or FOVM:
        net_prop = DualGCN(14, 'dual'); net_prop.load_state_dict(torch.load(f'{CACHE}/gcn_prop_f{FOLD}.pt')); net_prop.eval()
        net_prop.pos = float(np.mean(np.concatenate([g['y'] for g in graphs_for(enh, enh_aug, tr_idx, True)])))
        _n0 = len(rows)
        net_ns = run_gcn('GCN without seeds (automatic)', enh, enh_aug, 'dual', 'enriched', False)
        del rows[_n0:]
        for i in (test_idx if BASE else []):
            s_ = scribbles(i)
            g = build_graph(enh[i], s_, 400, 0.1, feats='enriched')
            keep('Proposed (GCN + CRF)', i, conv_crf(gcn_predict(net_prop, g, correct=True), enh[i], scr=s_))
            g2 = build_graph(enh[i], None, 400, 0.1, feats='enriched', use_seeds=False)
            keep('GCN without seeds (automatic)', i, gcn_predict(net_ns, g2, keep_seeds=False))
    else:
        net_prop = run_gcn('GCN-dual (enriched)', enh, enh_aug, 'dual', 'enriched', True, scenarios=SCEN, save=True,
                           crf_also='Proposed (GCN + CRF)')
        # original manuscript configuration: mean-intensity node feature + seed one-hot
        run_gcn('GCN-dual (original features)', enh, enh_aug, 'dual', 'basic', True, scenarios=('default',), save=True)
        if REP == 0:
            run_gcn('GCN-ChebNet only', enh, enh_aug, 'cheb', 'enriched', True, crf_also='ChebNet only + CRF')
            run_gcn('GCN-GraphSAGE only', enh, enh_aug, 'sage', 'enriched', True, crf_also='GraphSAGE only + CRF')
            net_ns = run_gcn('GCN without seeds (automatic)', enh, enh_aug, 'dual', 'enriched', False, save=True)
    if not FOVM:
        run_baselines()

    # ---------------------------------------------------------------- U-Nets (fully automatic / scribble-conditioned)
    raw = ds['img']

    def sin(img, s):
        return np.concatenate([img[None], geodesic_maps(img, s)], 0)
    if FOVM:
        un = UNet(1); un.load_state_dict(torch.load(f'{CACHE}/unet_auto_f{FOLD}.pt')); un.eval()
        us = UNet(3); us.load_state_dict(torch.load(f'{CACHE}/unet_scr_f{FOLD}.pt')); us.eval()
    else:
        t = time.time()
        Xtr = [raw[i][None] for i in tr_idx]; Ytr = [ds['msk'][i] for i in tr_idx]
        Xva = [raw[i][None] for i in va_idx]; Yva = [ds['msk'][i] for i in va_idx]
        un = train_unet(Xtr, Ytr, Xva, Yva, 1, seed=REP * 100 + FOLD, epochs=60, patience=10, steps_per_epoch=16)
        for i in test_idx:
            p = unet_predict(un, raw[i][None]); record('U-Net (2D, automatic)', i, p); keep('U-Net (2D, automatic)', i, p)
        log('U-Net auto', round(time.time() - t, 1)); flush()
        if REP == 0:
            torch.save(un.state_dict(), f'{CACHE}/unet_auto_f{FOLD}.pt')

        t = time.time()
        Xtr = [sin(raw[i], scribbles(i)) for i in tr_idx]; Xva = [sin(raw[i], scribbles(i)) for i in va_idx]
        us = train_unet(Xtr, Ytr, Xva, Yva, 3, seed=REP * 100 + FOLD + 1, epochs=60, patience=10, steps_per_epoch=16)
        for scen in SCEN:
            for i in test_idx:
                s = scribbles(i, scen)
                p = unet_predict(us, sin(raw[i], s)); record('Scribble U-Net (geodesic)', i, p, scen)
                keep('Scribble U-Net (geodesic)', i, p, scen)
        log('U-Net scribble', round(time.time() - t, 1)); flush()
        if REP == 0:
            torch.save(us.state_dict(), f'{CACHE}/unet_scr_f{FOLD}.pt')
            if not BASE:
                torch.save(net_prop.state_dict(), f'{CACHE}/gcn_prop_f{FOLD}.pt')
            with open(f'{RES}/pred/pred{"_base" if BASE else ""}_f{FOLD}.pkl', 'wb') as fh:
                pickle.dump(PRED, fh)


    # ---------------------------------------------------------------- mask-independent FOV and slice-wise analysis
    if REP == 0:
        t = time.time(); rows_fov = []
        test_p = sorted({pid[i] for i in test_idx})
        for p_ in test_p:
            d = np.load(ds['files'][p_])
            for s_ in range(4):
                im = d['full_img'][s_].astype(np.float32); gt = d['full_msk'][s_]
                H, W = im.shape; Hp, Wp = (H + 15) // 16 * 16, (W + 15) // 16 * 16
                imp = np.pad(im, ((0, Hp - H), (0, Wp - W)), mode='edge')
                rng = np.random.default_rng(5000 + int(p_.split('_')[1]) * 4 + s_)
                ref = d['full_msk'][1]
                scr = simulate_scribbles(gt, rng, ref_mask=ref)
                e_ = spcnn_apply(spc, wiener(im, psf))
                nseg = int(round(400 * H * W / 65536))
                g = build_graph(e_, scr, nseg, 0.1, feats='enriched')
                q = conv_crf(gcn_predict(net_prop, g, correct=True), e_, scr=scr)
                pu = unet_predict(un, imp[None])[:H, :W]
                for mname, pr in (('Proposed (GCN + CRF)', q), ('U-Net (2D, automatic)', pu)):
                    r = dict(pid=p_, sec=s_, method=mname, lesion=bool(gt.any()), fov='full slice', nseg=nseg)
                    if gt.any():
                        r.update(seg_metrics(pr > 0.5, gt))
                    else:
                        r.update(dict(fp_area=float((pr > 0.5).sum() * PIX_MM ** 2), fp_any=int((pr > 0.5).any())))
                    rows_fov.append(r)
        pd.DataFrame(rows_fov).to_csv(f'{RES}/fov_f{FOLD}.csv', index=False)
        log('FOV', round(time.time() - t, 1))

        t = time.time(); rows_sl = []
        for p_ in test_p:
            d = np.load(ds['files'][p_])
            xs = d['ext_x']; n = len(xs)
            for k in range(n):
                im = d['ext_img'][k].astype(np.float32); gt = d['ext_msk'][k]
                if not gt.any():
                    continue
                rng = np.random.default_rng(9000 + int(p_.split('_')[1]) * 1000 + k)
                scr = simulate_scribbles(gt, rng)
                e_ = spcnn_apply(spc, wiener(im, psf))
                g = build_graph(e_, scr, 400, 0.1, feats='enriched')
                q = conv_crf(gcn_predict(net_prop, g, correct=True), e_, scr=scr)
                pu = unet_predict(un, im[None]); ps_ = unet_predict(us, sin(im, scr))
                pos = (xs[k] - xs[0]) / max(xs[-1] - xs[0], 1)
                for mname, pr in (('Proposed (GCN + CRF)', q), ('U-Net (2D, automatic)', pu), ('Scribble U-Net (geodesic)', ps_)):
                    r = dict(pid=p_, slice=int(xs[k]), rel_pos=float(pos), method=mname)
                    r.update(seg_metrics(pr > 0.5, gt)); rows_sl.append(r)
        pd.DataFrame(rows_sl).to_csv(f'{RES}/slicewise_f{FOLD}.csv', index=False)
        log('slice-wise', round(time.time() - t, 1))

        # external lesion-free sections from 18 non-RC patients (different patients, harder negatives)
        rows_ext = []
        import glob as _g
        for f_ in [x for x in sorted(_g.glob(f'{DATA}/[DKA]*_*.npz')) if not os.path.basename(x).startswith(PREFIX + '_')]:
            d = np.load(f_); im = d['img'][0]
            pu = unet_predict(un, im[None])
            e_ = spcnn_apply(spc, wiener(im, psf))
            g = build_graph(e_, None, 400, 0.1, feats='enriched', use_seeds=False)
            pg = gcn_predict(net_ns, g, keep_seeds=False)
            for mname, pr in (('U-Net (2D, automatic)', pu), ('GCN without seeds (automatic)', pg)):
                rows_ext.append(dict(pid=os.path.basename(f_)[:-4], method=mname, fp_area=float((pr > 0.5).sum() * PIX_MM ** 2), fp_any=int((pr > 0.5).any())))
        if rows_ext:
            pd.DataFrame(rows_ext).to_csv(f'{RES}/extneg_unet_f{FOLD}.csv', index=False)

if REP == 0 and (EXTRAS or 'pre' in ONLY):
    # ---------------------------------------------------------------- pre-processing controls (M3)
    rows_pre = []
    variants = {}
    variants['none'] = (ds['img'], {i: AUG[i][0] for i in AUG})
    variants['PSF/Wiener only'] = (psf_img, psf_aug)
    _cf = f'{CACHE}/spcnn_raw_r{REP}_f{FOLD}.pt'
    if os.path.exists(_cf):
        spc_raw = SPCNN(); spc_raw.load_state_dict(torch.load(_cf)); spc_raw.eval()
    else:
        spc_raw = train_spcnn([ds['img'][i] for i in tr_idx], [ds['img'][i] for i in va_idx], seed=REP * 10 + FOLD + 5)
        torch.save(spc_raw.state_dict(), _cf)
    variants['SPCNN only'] = (np.stack([spcnn_apply(spc_raw, im) for im in ds['img']]),
                              {i: spcnn_apply(spc_raw, AUG[i][0]) for i in AUG})
    variants['PSF + SPCNN'] = (enh, enh_aug)
    for k in available_filters():
        cf = f'{CACHE}/filt_{k}.npy'
        if os.path.exists(cf):
            arr = np.load(cf)
        else:
            arr = np.stack([classical_filter(im, k) for im in ds['img']]); np.save(cf, arr)
        _af = f'{CACHE}/filt_aug_{k}_r{REP}.npz'
        _have = dict(np.load(_af)) if os.path.exists(_af) else {}
        _aug = {}
        for i in AUG:
            _aug[i] = _have[str(i)] if str(i) in _have else classical_filter(AUG[i][0], k)
            _have[str(i)] = _aug[i]
        np.savez(_af, **_have)
        variants[{'gaussian': 'Gaussian', 'bilateral': 'Bilateral', 'aniso': 'Anisotropic diffusion', 'nlm': 'Non-local means', 'bm3d': 'BM3D'}[k]] = (arr, _aug)
    for name, (A, Aaug) in variants.items():
        # image-quality measurements on held-out lesion sections: noise sigma and lesion/surround CNR
        for i in test_idx:
            if not ds['lesion'][i]:
                continue
            m = ds['msk'][i].astype(bool); ring = ndi.binary_dilation(m, iterations=8) & ~ndi.binary_dilation(m, iterations=2)
            a = A[i]
            cnr = abs(a[ring].mean() - a[m].mean()) / np.sqrt(0.5 * (a[ring].var() + a[m].var()) + 1e-12)
            rows_pre.append(dict(rep=REP, fold=FOLD, idx=int(i), variant=name, noise_sigma=float(restoration.estimate_sigma(a)), cnr=float(cnr)))
        n0 = len(rows)
        run_gcn('PRE|' + name, A, Aaug, 'dual', 'enriched', True, crf_also='PREPC|' + name)
        for r in rows[n0:]:
            r['variant'] = name
    pd.DataFrame(rows_pre).to_csv(f'{RES}/prequality_f{FOLD}.csv', index=False)
    flush()

if REP == 0 and (EXTRAS or 'sp' in ONLY):
    # ---------------------------------------------------------------- superpixel granularity (M7)
    for n_seg, comp in [(200, 0.1), (656, 0.1), (1000, 0.1), (400, 0.03), (400, 1.0)]:
        run_gcn(f'SP|N={n_seg}|c={comp}', enh, enh_aug, 'dual', 'enriched', True, n_seg=n_seg, compact=comp,
                crf_also=f'SP|N={n_seg}|c={comp}|CRF')
    flush()

if REP == 0 and (EXTRAS or 'sr' in ONLY):
    # ---------------------------------------------------------------- x2 super-resolution fidelity (Figure 3)
    import cv2
    from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

    class SRCNN(nn.Module):
        def __init__(self):
            super().__init__(); self.n = nn.Sequential(nn.Conv2d(1, 64, 9, padding=4), nn.ReLU(), nn.Conv2d(64, 32, 1), nn.ReLU(), nn.Conv2d(32, 1, 5, padding=2))
        def forward(self, x):
            return self.n(x)

    class VDSR(nn.Module):  # reduced-depth VDSR (10 layers x 32 filters, residual)
        def __init__(self, L=10, c=32):
            super().__init__()
            ls = [nn.Conv2d(1, c, 3, padding=1), nn.ReLU()]
            for _ in range(L - 2):
                ls += [nn.Conv2d(c, c, 3, padding=1), nn.ReLU()]
            ls += [nn.Conv2d(c, 1, 3, padding=1)]; self.n = nn.Sequential(*ls)
        def forward(self, x):
            return x + self.n(x)

    def train_sr(net, imgs, epochs=15):
        rng = np.random.default_rng(FOLD); opt = torch.optim.Adam(net.parameters(), 1e-4 if isinstance(net, SRCNN) else 1e-3)
        for ep in range(epochs):
            X, Y = [], []
            for im in imgs:
                bic, _ = degrade(im)
                for _ in range(4):
                    r, c = rng.integers(0, 256 - 48, 2); X.append(bic[r:r + 48, c:c + 48]); Y.append(im[r:r + 48, c:c + 48])
            X = torch.tensor(np.stack(X))[:, None]; Y = torch.tensor(np.stack(Y))[:, None]
            perm = torch.randperm(len(X))
            for i in range(0, len(X), 16):
                idb = perm[i:i + 16]; loss = F.mse_loss(net(X[idb]), Y[idb]); opt.zero_grad(); loss.backward(); opt.step()
        net.eval(); return net
    trn = [psf_img[i] for i in tr_idx]
    srcnn = train_sr(SRCNN(), trn); vdsr = train_sr(VDSR(), trn)
    rows_sr = []; examples = {}
    for i in test_idx:
        gt = psf_img[i]; bic, lr = degrade(gt)
        with torch.no_grad():
            outs = {'Bicubic': bic, 'SRCNN': srcnn(torch.tensor(bic)[None, None]).numpy()[0, 0],
                    'VDSR (reduced)': vdsr(torch.tensor(bic)[None, None]).numpy()[0, 0], 'SPCNN': spcnn_sr(spc, lr)}
        for k, o in outs.items():
            o = np.clip(o, 0, 1)
            rows_sr.append(dict(fold=FOLD, idx=int(i), method=k, psnr=psnr(gt, o, data_range=1), ssim=ssim(gt, o, data_range=1)))
        if len(examples) < 2 and ds['lesion'][i]:
            examples[int(i)] = {k: np.clip(v, 0, 1).astype(np.float16) for k, v in outs.items()} | {'LR': lr.astype(np.float16), 'HR': gt.astype(np.float16)}
    pd.DataFrame(rows_sr).to_csv(f'{RES}/sr_f{FOLD}.csv', index=False)
    with open(f'{RES}/pred/sr_examples_f{FOLD}.pkl', 'wb') as fh:
        pickle.dump(examples, fh)
    log('SR done')

flush()
log('UNIT DONE')

Writing /content/pal_code/run_seg.py


`run_sam.py` — zero-shot SAM / MedSAM baselines

In [ ]:
%%writefile /content/pal_code/run_sam.py
"""Zero-shot promptable baselines (M4): SAM ViT-B and MedSAM ViT-B, prompts derived from the SAME simulated
scribbles that the graph methods receive (information-matched interaction budget).
 - SAM (points): up to 3 positive points from the lesion scribble + up to 3 negative points from the background scribble
 - SAM (box) / MedSAM (box): bounding box of the lesion scribble enlarged by a fixed 10-pixel (3 mm) margin
No lesion scribble -> no prompt -> empty prediction. Image embeddings are cached."""
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import sys, os, zlib, time, pickle
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np, pandas as pd, torch
from pal.core import *
from transformers import SamModel, SamProcessor

torch.set_num_threads(int(os.environ.get('NTH', '1')))
RES = PAL_WORK + '/results'; CACHE = PAL_WORK + '/cache'
os.makedirs(f'{CACHE}/sam', exist_ok=True)
ds = load_dataset(); pid = ds['pid']; REP = 0
fold_of = make_folds(ds, 4, seed=0)
ref_mask = {pid[i]: ds['msk'][i] for i in range(len(pid)) if ds['sec'][i] == 1}
SCEN = ('default', 'density05', 'density30', 'boundary', 'mislabel10', 'mislabel20', 'click', 'missing', 'opA', 'opB', 'opC')
part = int(sys.argv[1]) if len(sys.argv) > 1 else 0; nparts = int(sys.argv[2]) if len(sys.argv) > 2 else 1


def scribbles(i, mode='default', rep=REP, salt=0):
    rng = np.random.default_rng(1_000_003 * rep + 7919 * i + 104729 * salt + zlib.crc32(mode.encode()) % 9973)
    return simulate_scribbles(ds['msk'][i], rng, mode=mode, ref_mask=ref_mask[pid[i]])


def to_rgb(img):
    lo, hi = np.percentile(img, [0.5, 99.5])
    u = np.clip((img - lo) / max(hi - lo, 1e-6), 0, 1)
    return (np.stack([u] * 3, -1) * 255).astype(np.uint8)


def prompts(scr):
    les = np.argwhere(scr == 1); bg = np.argwhere(scr == 0)
    if len(les) == 0:
        return None, None, None
    sel = lambda a, k: a[np.linspace(0, len(a) - 1, min(k, len(a))).astype(int)]
    pp = sel(les, 3); nn_ = sel(bg, 3) if len(bg) else np.zeros((0, 2), int)
    pts = [[float(c), float(r)] for r, c in pp] + [[float(c), float(r)] for r, c in nn_]
    lab = [1] * len(pp) + [0] * len(nn_)
    r0, c0 = les.min(0); r1, c1 = les.max(0)
    box = [float(max(c0 - 10, 0)), float(max(r0 - 10, 0)), float(min(c1 + 10, 255)), float(min(r1 + 10, 255))]
    return pts, lab, box


rows = []
for mname, hub in (('SAM', 'facebook/sam-vit-base'), ('MedSAM', 'wanglab/medsam-vit-base')):
    proc = SamProcessor.from_pretrained(hub); model = SamModel.from_pretrained(hub).eval()
    t0 = time.time()
    for i in range(len(pid)):
        if i % nparts != part:
            continue
        rgb = to_rgb(ds['img'][i])
        cf = f'{CACHE}/sam/{mname}_{i}.pt'
        base = proc(images=rgb, return_tensors='pt')
        if os.path.exists(cf):
            emb = torch.load(cf)
        else:
            with torch.no_grad():
                emb = model.get_image_embeddings(base['pixel_values'])
            torch.save(emb, cf)
        for scen in SCEN:
            scr = scribbles(i, scen)
            pts, lab, box = prompts(scr)
            variants = [('points', 'SAM (points)'), ('box', 'SAM (box)')] if mname == 'SAM' else [('box', 'MedSAM (box)')]
            for kind, label in variants:
                if pts is None:
                    prob = np.zeros((256, 256), np.float32)
                else:
                    kw = dict(input_points=[[pts]], input_labels=[[lab]]) if kind == 'points' else dict(input_boxes=[[box]])
                    inp = proc(images=rgb, return_tensors='pt', **kw)
                    inp.pop('pixel_values'); inp['image_embeddings'] = emb
                    with torch.no_grad():
                        out = model(**inp, multimask_output=False)
                    m = proc.image_processor.post_process_masks(out.pred_masks, inp['original_sizes'], inp['reshaped_input_sizes'], binarize=False)[0]
                    prob = torch.sigmoid(m[0, 0]).numpy()
                pred = prob > 0.5
                r = dict(rep=0, fold=fold_of[pid[i]], idx=i, pid=pid[i], sec=int(ds['sec'][i]), lesion=bool(ds['lesion'][i]), method=label, scenario=scen)
                if ds['lesion'][i]:
                    r.update(seg_metrics(pred, ds['msk'][i]))
                else:
                    r.update(dict(fp_pixels=int(pred.sum()), fp_area=float(pred.sum() * PIX_MM ** 2), fp_any=int(pred.sum() > 0)))
                rows.append(r)
                if scen == 'default':
                    os.makedirs(f'{RES}/pred', exist_ok=True)
                    np.save(f'{CACHE}/sam/pred_{label.replace(" ", "_").replace("(", "").replace(")", "")}_{i}.npy', prob.astype(np.float16))
        if i % 20 == 0:
            print(mname, i, round(time.time() - t0), flush=True)
    del model
    pd.DataFrame(rows).to_csv(f'{RES}/sam_part{part}.csv', index=False)
print('SAM DONE', flush=True)

Writing /content/pal_code/run_sam.py


`run_det.py` — Faster R-CNN lesion flagging

In [ ]:
%%writefile /content/pal_code/run_det.py
"""Stage 3 (revised, M2/M9): automatic lesion flagging with Faster R-CNN, ResNet-50-FPN backbone pre-trained on
COCO (torchvision), fine-tuned on the ENHANCED IMAGE ALONE (no scribble-derived input), so that it can flag
lesions the operator did not mark. Evaluated per fold (repeat 0) with patient-level folds.
usage: python run_det.py FOLD"""
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import sys, os, time, pickle
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np, pandas as pd, torch, torchvision
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pal.core import *

torch.set_num_threads(int(os.environ.get('NTH', '1')))
FOLD = int(sys.argv[1]); EPOCHS = int(os.environ.get('DET_EPOCHS', '12'))
RES = PAL_WORK + '/results'; CACHE = PAL_WORK + '/cache'
os.makedirs(f'{RES}/pred', exist_ok=True); os.makedirs(CACHE, exist_ok=True)
ds = load_dataset(); pid = ds['pid']
fold_of = make_folds(ds, 4, seed=0)
test_idx = np.where([fold_of[p] == FOLD for p in pid])[0]
train_p = sorted({p for p in pid if fold_of[p] != FOLD})
tr_p, va_p = split_train_val(train_p, seed=FOLD)
tr_idx = np.where(np.isin(pid, tr_p + va_p))[0]  # detector: fixed epoch budget, no early stopping

# enhanced images: same fold-specific PSF + SPCNN as the segmentation stage
psf = np.load(f'{CACHE}/psf_f{FOLD}.npy')
spc = SPCNN(); spc.load_state_dict(torch.load(f'{CACHE}/spcnn_f{FOLD}.pt')); spc.eval()
rng = np.random.default_rng(FOLD); t0 = time.time()
enh = {i: spcnn_apply(spc, wiener(ds['img'][i], psf)) for i in np.concatenate([tr_idx, test_idx])}


def target(i, m=None):
    m = ds['msk'][i] if m is None else m
    b = boxes_from_mask(m, min_area=5)
    return dict(boxes=torch.tensor(b).reshape(-1, 4), labels=torch.ones(len(b), dtype=torch.int64))


from torchvision.models.detection.image_list import ImageList
# COCO-pre-trained ResNet-50-FPN kept frozen (CPU budget); RPN and RoI heads are fine-tuned on cached FPN features.
model = fasterrcnn_resnet50_fpn(weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1, min_size=256, max_size=256,
                                trainable_backbone_layers=0, box_detections_per_img=20)
model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features, 2)
for p in model.backbone.parameters():
    p.requires_grad = False
model.eval()


def feats(img):
    x = torch.tensor(np.repeat(img[None], 3, 0)).float()
    with torch.no_grad():
        il, _ = model.transform([x])
        f = model.backbone(il.tensors)
    return {k: v.half() for k, v in f.items()}, il.image_sizes


cache = {}
for i in np.concatenate([tr_idx, test_idx]):
    cache[(int(i), 0)] = feats(enh[i])
    if i in tr_idx:
        cache[(int(i), 1)] = feats(enh[i][:, ::-1].copy())
print('features cached', time.time() - t0, flush=True)


def tgt(i, flip):
    m = ds['msk'][i][:, ::-1] if flip else ds['msk'][i]
    b = boxes_from_mask(m, min_area=5)
    return dict(boxes=torch.tensor(b).reshape(-1, 4), labels=torch.ones(len(b), dtype=torch.int64))


params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.SGD(params, lr=0.01, momentum=0.9, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.MultiStepLR(opt, [int(EPOCHS * 0.7)], 0.1)
keys = [k for k in cache if k[0] in set(tr_idx.tolist())]
for ep in range(EPOCHS):
    model.rpn.train(); model.roi_heads.train(); tot = 0
    perm = rng.permutation(len(keys))
    for s_ in range(0, len(perm), 4):
        ks = [keys[j] for j in perm[s_:s_ + 4]]
        F_ = {lvl: torch.cat([cache[k][0][lvl] for k in ks]).float() for lvl in cache[ks[0]][0]}
        sizes = [cache[k][1][0] for k in ks]
        il = ImageList(torch.zeros(len(ks), 3, 256, 256), sizes)
        tg = [tgt(*k) for k in ks]
        props, l1 = model.rpn(il, F_, tg)
        _, l2 = model.roi_heads(F_, props, il.image_sizes, tg)
        loss = sum(l1.values()) + sum(l2.values())
        opt.zero_grad(); loss.backward(); opt.step(); tot += loss.item()
    sched.step()
    print(f'fold {FOLD} epoch {ep} loss {tot:.3f} t {time.time() - t0:.0f}s', flush=True)
model.eval(); model.roi_heads.score_thresh = 0.01
out = {}
with torch.no_grad():
    for i in test_idx:
        f, sz = cache[(int(i), 0)]
        F_ = {k: v.float() for k, v in f.items()}
        il = ImageList(torch.zeros(1, 3, 256, 256), sz)
        props, _ = model.rpn(il, F_)
        det, _ = model.roi_heads(F_, props, il.image_sizes)
        out[int(i)] = dict(boxes=det[0]['boxes'].numpy(), scores=det[0]['scores'].numpy())
with open(f'{RES}/pred/det_f{FOLD}.pkl', 'wb') as fh:
    pickle.dump(out, fh)
torch.save(model.state_dict(), f'{CACHE}/frcnn_f{FOLD}.pt')
print('DET DONE', FOLD, time.time() - t0, flush=True)

Writing /content/pal_code/run_det.py


`run_timing.py` — inference time per stage

In [ ]:
%%writefile /content/pal_code/run_timing.py
"""Inference time per 256 x 256 section (2 vCPU, no GPU), fold-0 models, 30 held-out sections."""
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import os, sys, time, glob, zlib
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np, pandas as pd, torch
from pal.core import *
torch.set_num_threads(int(os.environ.get('NTH', '2')))
CACHE = PAL_WORK + '/cache'; RES = PAL_WORK + '/results'
ds = load_dataset(); pid = ds['pid']; fo = make_folds(ds, 4, 0)
test = [i for i in range(len(pid)) if fo[pid[i]] == 0][:30]
ref = {pid[i]: ds['msk'][i] for i in range(len(pid)) if ds['sec'][i] == 1}
psf = np.load(f'{CACHE}/psf_f0.npy'); spc = SPCNN(); spc.load_state_dict(torch.load(f'{CACHE}/spcnn_f0.pt')); spc.eval()
z = np.load(f'{CACHE}/pre_r0_f0.npz', allow_pickle=True)
tr_ids = [i for i in range(len(pid)) if fo[pid[i]] != 0][:40]
gs = [build_graph(z['enh'][i], simulate_scribbles(ds['msk'][i], np.random.default_rng(i), ref_mask=ref[pid[i]]), mask=ds['msk'][i]) for i in tr_ids]
net = DualGCN(gs[0]['X'].shape[1]); net.load_state_dict(torch.load(f'{CACHE}/gcn_prop_f0.pt')); net.eval(); net.pos = 0.017
un = UNet(1); un.load_state_dict(torch.load(f'{CACHE}/unet_auto_f0.pt')); un.eval()
us = UNet(3); us.load_state_dict(torch.load(f'{CACHE}/unet_scr_f0.pt')); us.eval()
T = {k: [] for k in ['PSF/Wiener deconvolution', 'SPCNN pre-filter', 'SLIC + graph construction (incl. features)', 'Dual GCN inference',
                     'Prior correction + ConvCRF refinement', 'Total proposed pipeline', 'Scribble U-Net (geodesic maps + inference)', 'U-Net (automatic) inference']}
# warm-up (not timed)
_s = simulate_scribbles(ds['msk'][test[0]], np.random.default_rng(0), ref_mask=ref[pid[test[0]]])
_e = spcnn_apply(spc, wiener(ds['img'][test[0]], psf)); conv_crf(prior_correct(gcn_predict(net, build_graph(_e, _s, 400, 0.1, feats='enriched')), 0.017), _e, scr=_s)
for i in test:
    s = simulate_scribbles(ds['msk'][i], np.random.default_rng(i), ref_mask=ref[pid[i]])
    t0 = time.perf_counter(); w = wiener(ds['img'][i], psf); t1 = time.perf_counter()
    e = spcnn_apply(spc, w); t2 = time.perf_counter()
    g = build_graph(e, s, 400, 0.1, feats='enriched'); t3 = time.perf_counter()
    p = gcn_predict(net, g); t4 = time.perf_counter()
    q = conv_crf(prior_correct(p, 0.017), e, scr=s); t5 = time.perf_counter()
    T['PSF/Wiener deconvolution'].append(t1 - t0); T['SPCNN pre-filter'].append(t2 - t1); T['SLIC + graph construction (incl. features)'].append(t3 - t2)
    T['Dual GCN inference'].append(t4 - t3); T['Prior correction + ConvCRF refinement'].append(t5 - t4); T['Total proposed pipeline'].append(t5 - t0)
    t0 = time.perf_counter(); unet_predict(us, np.concatenate([ds['img'][i][None], geodesic_maps(ds['img'][i], s)])); T['Scribble U-Net (geodesic maps + inference)'].append(time.perf_counter() - t0)
    t0 = time.perf_counter(); unet_predict(un, ds['img'][i][None]); T['U-Net (automatic) inference'].append(time.perf_counter() - t0)
rows = [{'Stage': k, 'Time per section (ms)': f'{1000 * np.mean(v):.1f} ± {1000 * np.std(v):.1f}'} for k, v in T.items()]
# SAM/MedSAM: one image embedding + one prompt decode
try:
    from transformers import SamModel, SamProcessor
    proc = SamProcessor.from_pretrained('wanglab/medsam-vit-base'); model = SamModel.from_pretrained('wanglab/medsam-vit-base').eval()
    tt = []
    for i in test[:5]:
        rgb = (np.stack([ds['img'][i]] * 3, -1) * 255).astype(np.uint8)
        t0 = time.perf_counter()
        with torch.no_grad():
            emb = model.get_image_embeddings(proc(images=rgb, return_tensors='pt')['pixel_values'])
            inp = proc(images=rgb, input_boxes=[[[60., 60., 120., 120.]]], return_tensors='pt'); inp.pop('pixel_values'); inp['image_embeddings'] = emb
            model(**inp, multimask_output=False)
        tt.append(time.perf_counter() - t0)
    rows.append({'Stage': 'MedSAM / SAM ViT-B (embedding + box decode)', 'Time per section (ms)': f'{1000 * np.mean(tt):.1f} ± {1000 * np.std(tt):.1f}'})
except Exception as ex:
    print('sam timing failed', ex)
rows.append({'Stage': 'Operator time to draw scribbles', 'Time per section (ms)': 'not measured (no human-operator study; see Section 4.3)'})
pd.DataFrame(rows).to_csv(f'{RES}/tables/T_timing.csv', index=False)
print(pd.DataFrame(rows))

Writing /content/pal_code/run_timing.py


`precompute_filters.py` — classical pre-filter controls

In [ ]:
%%writefile /content/pal_code/precompute_filters.py
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import sys, os; sys.path.insert(0, os.path.dirname(__file__))
from pal.core import *
os.makedirs(PAL_WORK + '/cache', exist_ok=True)
ds = load_dataset()
for k in available_filters():
    cf = PAL_WORK + f'/cache/filt_{k}.npy'
    if not os.path.exists(cf):
        np.save(cf, np.stack([classical_filter(im, k) for im in ds['img']])); print(k, flush=True)

Writing /content/pal_code/precompute_filters.py


`analyze_helpers.py` — figure saving helper

In [ ]:
%%writefile /content/pal_code/analyze_helpers.py
import os as _os_pal
PAL_WORK = _os_pal.environ.get('PAL_WORK', '/content/pal_work')
import matplotlib.pyplot as plt
FIG = PAL_WORK + '/figures'
INK = '#0b0b0b'; MUTED = '#52514e'


def save(fig, name):
    import os
    os.makedirs(FIG, exist_ok=True)
    for ext in ('png', 'tiff', 'pdf', 'svg', 'eps'):
        kw = dict(dpi=600) if ext in ('png', 'tiff') else {}
        if ext == 'tiff':
            kw['pil_kwargs'] = {'compression': 'tiff_lzw'}
        fig.savefig(f'{FIG}/{name}.{ext}', bbox_inches='tight', **kw)
    plt.close(fig)

Writing /content/pal_code/analyze_helpers.py


`export_scribbles.py` — export of the simulated scribble maps

In [ ]:
%%writefile /content/pal_code/export_scribbles.py
"""Export the exact simulated scribble maps used in every experiment (reviewer minor comment 16).
Writes scribbles/<scenario>/rep<r>/<pid>_s<sec>.png  (0 = background scribble, 1 = lesion scribble, 255 = unlabelled)."""
import os, sys, zlib
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np
from PIL import Image
from pal.core import load_dataset, simulate_scribbles

SCEN = ('default', 'density05', 'density30', 'boundary', 'mislabel10', 'mislabel20', 'click', 'missing', 'opA', 'opB', 'opC')
out = sys.argv[1] if len(sys.argv) > 1 else './scribbles'
ds = load_dataset(); pid = ds['pid']
ref = {pid[i]: ds['msk'][i] for i in range(len(pid)) if ds['sec'][i] == 1}
for rep in (0, 1, 2):
    for scen in (SCEN if rep == 0 else ('default',)):
        d = f'{out}/{scen}/rep{rep}'; os.makedirs(d, exist_ok=True)
        for i in range(len(pid)):
            rng = np.random.default_rng(1_000_003 * rep + 7919 * i + zlib.crc32(scen.encode()) % 9973)
            s = simulate_scribbles(ds['msk'][i], rng, mode=scen, ref_mask=ref[pid[i]])
            Image.fromarray(np.where(s < 0, 255, s.astype(np.int16)).astype(np.uint8)).save(f'{d}/{pid[i]}_s{ds["sec"][i]}.png')
print('done')

Writing /content/pal_code/export_scribbles.py


`export_nnunet.py` — nnU-Net v2 dataset export

In [ ]:
%%writefile /content/pal_code/export_nnunet.py
"""Convert the extracted DOLCHID sections into an nnU-Net v2 raw dataset (2D) with the SAME patient-level folds,
so that the official nnU-Net can be added on a GPU machine (reviewer comment M4).

    python export_nnunet.py /path/to/nnUNet_raw
    nnUNetv2_plan_and_preprocess -d 501 --verify_dataset_integrity
    # copy splits_final.json written by this script into nnUNet_preprocessed/Dataset501_DOLCHIDRC/
    for f in 0 1 2 3; do nnUNetv2_train 501 2d $f; done
    nnUNetv2_find_best_configuration 501 -c 2d
Then evaluate the fold predictions with pal.core.seg_metrics exactly as in run_seg.py.
"""
import os, sys, json
sys.path.insert(0, os.path.dirname(__file__))
import numpy as np
from PIL import Image
from pal.core import load_dataset, make_folds

out = sys.argv[1] if len(sys.argv) > 1 else './nnUNet_raw'
D = os.path.join(out, 'Dataset501_DOLCHIDRC'); os.makedirs(f'{D}/imagesTr', exist_ok=True); os.makedirs(f'{D}/labelsTr', exist_ok=True)
ds = load_dataset(); fold_of = make_folds(ds, 4, seed=0)
cases = []
for i in range(len(ds['img'])):
    cid = f"{ds['pid'][i]}_s{ds['sec'][i]}"
    Image.fromarray((ds['img'][i] * 255).astype(np.uint8)).save(f'{D}/imagesTr/{cid}_0000.png')
    Image.fromarray(ds['msk'][i].astype(np.uint8)).save(f'{D}/labelsTr/{cid}.png')
    cases.append((cid, fold_of[ds['pid'][i]]))
json.dump({'channel_names': {'0': 'CBCT'}, 'labels': {'background': 0, 'lesion': 1}, 'numTraining': len(cases), 'file_ending': '.png'},
          open(f'{D}/dataset.json', 'w'), indent=1)
splits = [{'train': [c for c, f in cases if f != k], 'val': [c for c, f in cases if f == k]} for k in range(4)]
json.dump(splits, open(f'{D}/splits_final.json', 'w'), indent=1)
print('written', len(cases), 'cases to', D)

Writing /content/pal_code/export_nnunet.py


In [ ]:
# check that every file was written and compiles
import py_compile, glob
files = sorted(glob.glob(f'{CODE}/**/*.py', recursive=True))
for f in files:
    py_compile.compile(f, doraise=True)
print(len(files), 'files compiled OK:', ', '.join(os.path.relpath(f, CODE) for f in files))

15 files compiled OK: analyze.py, analyze_helpers.py, export_nnunet.py, export_scribbles.py, extract_sections.py, fig_qual.py, fig_schematics.py, httpfile.py, pal/__init__.py, pal/core.py, precompute_filters.py, run_det.py, run_sam.py, run_seg.py, run_timing.py


## Step 4 — Packages


In [ ]:
import subprocess, sys, importlib
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'bm3d'], check=False)
try:
    import bm3d
except Exception as ex:
    print('bm3d import failed:', ex, '-> installing OpenBLAS and retrying')
    subprocess.run('apt-get -qq update && apt-get -qq install -y libopenblas-dev', shell=True)
    try:
        importlib.invalidate_caches(); import bm3d
    except Exception as ex2:
        print('bm3d still unavailable (', ex2, ') - the BM3D pre-filter control will be skipped; everything else runs.')
need = {'torch': '2.0', 'torchvision': '0.15', 'numpy': '1.24', 'scipy': '1.10', 'pandas': '1.5', 'skimage': '0.19',
        'sklearn': '1.2', 'statsmodels': '0.14', 'cv2': '4.5', 'nibabel': '4.0', 'matplotlib': '3.6', 'transformers': '4.40', 'requests': '2.25'}
from packaging.version import Version
bad = []
for m, v in need.items():
    try:
        mod = importlib.import_module(m); ver = getattr(mod, '__version__', '0')
        ok = Version(ver.split('+')[0]) >= Version(v)
        print(f'{m:13s} {ver:14s}', 'OK' if ok else f'TOO OLD (need >= {v})')
        if not ok: bad.append(m)
    except Exception as ex:
        print(f'{m:13s} MISSING ({ex})'); bad.append(m)
pipname = {'skimage': 'scikit-image', 'sklearn': 'scikit-learn', 'cv2': 'opencv-python-headless'}
if bad:
    print('\nRun:  !pip install -U ' + ' '.join(pipname.get(b, b) for b in bad) + '   then Runtime -> Restart session and re-run Steps 1-4.')

torch         2.14.1+cpu     OK
torchvision   0.29.1+cpu     OK
numpy         2.4.6          OK
scipy         1.16.3         OK
pandas        2.2.3          OK
skimage       0.25.2         OK
sklearn       1.6.1          OK
statsmodels   0.15.0         OK
cv2           5.0.0          OK
nibabel       5.4.2          OK
matplotlib    3.10.0         OK
transformers  5.22.1         OK
requests      2.32.4         OK


In [ ]:
#@title Data source  (re-run this cell after every reconnect)
import os
for k in ('PAL_DOLCHID_URL', 'PAL_DOLCHID_ZIP'):
    os.environ.pop(k, None)
os.environ['PAL_DOLCHID_DIR'] = DATA_ROOT
os.environ['PAL_IMAGE_DIR']   = CBCT_IMG
os.environ['PAL_LABEL_DIR']   = CBCT_LBL
os.environ['PAL_DATA']        = f'{WORK}/data'
os.environ['PAL_PREFIX']      = CASE_PREFIX
assert os.path.isdir(CBCT_IMG) and os.path.isdir(CBCT_LBL), 'cbct_image / cbct_label not found inside DOLCHID'
print({k: v for k, v in os.environ.items() if k.startswith('PAL_')})


{'PAL_WORK': '/content/drive/MyDrive/DOLCHID', 'PAL_DATA': '/content/drive/MyDrive/DOLCHID/data', 'PAL_PREFIX': 'AME', 'PAL_DOLCHID_DIR': '/content/drive/MyDrive/DOLCHID', 'PAL_IMAGE_DIR': '/content/drive/MyDrive/DOLCHID/cbct_image', 'PAL_LABEL_DIR': '/content/drive/MyDrive/DOLCHID/cbct_label'}


In [ ]:
import sys, zipfile, collections
sys.path.insert(0, CODE)
from extract_sections import Source
src = Source()
names = list(src.members) + list(src.files)
print('files found:', len(names))
kinds = collections.Counter(n.split('_')[0] for n in names if n.endswith(('.nii.gz', '.nii')))
print('by prefix  :', dict(kinds))
print('examples   :', [n for n in names if n.startswith('RC_1_')][:4])
miss = [f'RC_{k}' for k in range(1, 55) if not any(n.startswith(f'RC_{k}_CBCT_Label') for n in names)]
print('RC patients without a label file:', miss or 'none')

files found: 50
by prefix  : {'AME': 50}
examples   : []
RC patients without a label file: ['RC_1', 'RC_2', 'RC_3', 'RC_4', 'RC_5', 'RC_6', 'RC_7', 'RC_8', 'RC_9', 'RC_10', 'RC_11', 'RC_12', 'RC_13', 'RC_14', 'RC_15', 'RC_16', 'RC_17', 'RC_18', 'RC_19', 'RC_20', 'RC_21', 'RC_22', 'RC_23', 'RC_24', 'RC_25', 'RC_26', 'RC_27', 'RC_28', 'RC_29', 'RC_30', 'RC_31', 'RC_32', 'RC_33', 'RC_34', 'RC_35', 'RC_36', 'RC_37', 'RC_38', 'RC_39', 'RC_40', 'RC_41', 'RC_42', 'RC_43', 'RC_44', 'RC_45', 'RC_46', 'RC_47', 'RC_48', 'RC_49', 'RC_50', 'RC_51', 'RC_52', 'RC_53', 'RC_54']


In [ ]:
import subprocess, os, sys, re, glob, time, numpy as np
env = os.environ.copy(); env.update(PAL_WORK=WORK, PAL_DATA=f'{WORK}/data', PAL_PREFIX=CASE_PREFIX, PYTHONUNBUFFERED='1')

def _ids(folder, kind):
    out = set()
    for f in glob.glob(f'{folder}/*_CBCT_{kind}.nii*'):
        m = re.match(r'([A-Za-z]+_\d+)_CBCT', os.path.basename(f))
        if m: out.add(m.group(1))
    return out
have = _ids(CBCT_IMG, 'Image') & _ids(CBCT_LBL, 'Label')       # IDs that have BOTH image and label
num = lambda s: int(s.split('_')[1])

MAIN_IDS  = sorted([p for p in have if p.startswith(CASE_PREFIX + '_')], key=num)          # main study group (4 sections each)
OTHER_IDS = [f'{p}_{k}' for p in ('DC', 'KCOT', 'AME') if p != CASE_PREFIX
             for k in range(1, 7) if f'{p}_{k}' in have]                                   # external lesion-free cases (1 section each)
print(f'main group {CASE_PREFIX}: {len(MAIN_IDS)} patients | external negatives: {len(OTHER_IDS)} (of other groups, max 18)')
assert MAIN_IDS, f'no {CASE_PREFIX}_* image+label pairs found in cbct_image / cbct_label'

# remove stale files of the main group that were extracted earlier as 1-section "external negatives"
for pid_ in MAIN_IDS:
    f = f'{WORK}/data/{pid_}.npz'
    if os.path.exists(f):
        try:
            ok = np.load(f)['img'].shape[0] == 4
        except Exception:
            ok = False
        if not ok:
            os.remove(f); print('removed stale', f)

for kind, ids in (('RC', MAIN_IDS), ('OTHER', OTHER_IDS)):      # 'RC' = 4-section extraction mode of extract_sections.py
    if not ids: continue
    t0 = time.time()
    p = subprocess.run([sys.executable, 'extract_sections.py', kind, *ids], cwd=CODE, env=env)
    print(kind, 'finished with code', p.returncode, 'in %.1f min' % ((time.time() - t0) / 60))
    if p.returncode:
        raise SystemExit('Extraction failed - see the message above and the Appendix (section B).')


main group AME: 25 patients | external negatives: 0 (of other groups, max 18)
removed stale /content/drive/MyDrive/DOLCHID/data/AME_1.npz
removed stale /content/drive/MyDrive/DOLCHID/data/AME_2.npz
removed stale /content/drive/MyDrive/DOLCHID/data/AME_3.npz
removed stale /content/drive/MyDrive/DOLCHID/data/AME_4.npz
removed stale /content/drive/MyDrive/DOLCHID/data/AME_5.npz
removed stale /content/drive/MyDrive/DOLCHID/data/AME_6.npz
RC finished with code 0 in 7.3 min


In [ ]:
import glob, shutil, numpy as np, importlib, os, sys
P = CASE_PREFIX
n_main = len(glob.glob(f'{WORK}/data/{P}_*.npz'))
n_ot = len([f for f in glob.glob(f'{WORK}/data/[DKA]*_*.npz') if not os.path.basename(f).startswith(P + '_')])
print(f'{P} patients:', n_main, '| external negatives:', n_ot)
assert n_main > 0, f'No {P}_*.npz in {WORK}/data - run the extraction cell first.'
if COPY_DATA_TO_LOCAL_DISK:
    shutil.copytree(f'{WORK}/data', '/content/pal_data', dirs_exist_ok=True)   # local copy for fast reads
    os.environ['PAL_DATA'] = '/content/pal_data'
else:
    os.environ['PAL_DATA'] = f'{WORK}/data'
os.environ['PAL_PREFIX'] = P
sys.path.insert(0, CODE)
import pal.core as core; importlib.reload(core)
ds = core.load_dataset()
print('sections:', ds['img'].shape, '| lesion sections:', int(ds['lesion'].sum()), '| lesion-free:', int((~ds['lesion']).sum()))
print('intensity range: %.3f-%.3f | NaNs: %d' % (ds['img'].min(), ds['img'].max(), int(np.isnan(ds['img']).sum())))
folds = core.make_folds(ds, 4, 0); print('patients per fold:', np.bincount(list(folds.values())))
assert ds['img'].shape[0] == 4 * n_main, 'Data incomplete - re-run the extraction cell.'
if n_main != 54:
    print(f'NOTE: the article used 54 RC patients (216 sections); this run uses {n_main} {P} patients ({4 * n_main} sections), so results will differ from the article.')


AME patients: 25 | external negatives: 0
sections: (100, 256, 256) | lesion sections: 75 | lesion-free: 25
intensity range: 0.000-1.000 | NaNs: 0
patients per fold: [7 6 6 6]
NOTE: the article used 54 RC patients (216 sections); this run uses 25 AME patients (100 sections), so results will differ from the article.


## Step 7 — Smoke test

In [ ]:
import subprocess, os, sys, time, shutil
shutil.rmtree('/content/pal_smoke', ignore_errors=True)
env = os.environ.copy(); env.update(PAL_WORK=WORK, SMOKE='1', RESDIR='/content/pal_smoke/results', CACHEDIR='/content/pal_smoke/cache',
                                    NTH=str(multiprocessing.cpu_count()), PYTHONUNBUFFERED='1')
t0 = time.time()
p = subprocess.run(f'{sys.executable} run_seg.py 0 0', shell=True, cwd=CODE, env=env)
print('smoke test', 'PASSED' if p.returncode == 0 else 'FAILED', 'in %.1f min' % ((time.time() - t0) / 60))

smoke test PASSED in 11.8 min


## Step 8 — Experiments

In [ ]:
import subprocess, os, sys, time, threading, concurrent.futures as cf, signal
LOGDIR = f'{WORK}/results/joblogs'; DONE = f'{LOGDIR}/done'; os.makedirs(DONE, exist_ok=True)
_PROCS = set(); _LOCK = threading.Lock()

def _name(cmd):
    return cmd.replace('python ', '').replace('.py', '').replace(' --only=', '_').replace(' ', '_')

def run(cmd, name=None, extra_env=None, nth=None, echo=True, force=False):
    name = name or _name(cmd); ok = f'{DONE}/{name}.ok'; log = f'{LOGDIR}/{name}.log'
    if os.path.exists(ok) and not force:
        print(f'[skip] {name} (done {open(ok).read().strip()})', flush=True); return 0
    env = os.environ.copy(); env.pop('SMOKE', None)
    env.update(PAL_WORK=WORK, NTH=str(nth or NTH_PER_JOB), PYTHONUNBUFFERED='1')
    if extra_env: env.update(extra_env)
    t0 = time.time(); print(f'[start] {name}  ({time.strftime("%H:%M:%S")})', flush=True)
    with open(log, 'a') as fh:
        fh.write(f'\n===== {time.ctime()}  {cmd}\n')
        p = subprocess.Popen(cmd.replace('python ', sys.executable + ' ', 1), shell=True, cwd=CODE, env=env,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, start_new_session=True)
        with _LOCK: _PROCS.add(p)
        for line in p.stdout:
            fh.write(line); fh.flush()
            if echo: print(f'  {line}', end='', flush=True)
        rc = p.wait()
        with _LOCK: _PROCS.discard(p)
    dt = (time.time() - t0) / 60
    if rc == 0:
        open(ok, 'w').write(time.ctime()); print(f'[done] {name} in {dt:.1f} min', flush=True)
    else:
        tail = ''.join(open(log, errors='replace').readlines()[-30:])
        print(f'[FAILED rc={rc}] {name} after {dt:.1f} min - log: {log}\n----- last lines -----\n{tail}', flush=True)
    return rc

def run_group(cmds, parallel=None, heartbeat_min=10):
    parallel = max(1, parallel or PARALLEL_JOBS)
    if parallel == 1:
        try:
            return [run(c) for c in cmds]
        except KeyboardInterrupt:
            _kill_all(); raise
    stop = threading.Event()
    def beat():                      # prints the last log line of every running job every few minutes
        while not stop.wait(heartbeat_min * 60):
            with _LOCK: running = len(_PROCS)
            print(f'--- {time.strftime("%H:%M")} | {running} job(s) running', flush=True)
            for c in cmds:
                log = f'{LOGDIR}/{_name(c)}.log'
                if os.path.exists(log) and not os.path.exists(f'{DONE}/{_name(c)}.ok'):
                    lines = open(log, errors='replace').read().strip().splitlines()
                    print(f'    {_name(c):28s} {lines[-1] if lines else ""}', flush=True)
    threading.Thread(target=beat, daemon=True).start()
    ex = cf.ThreadPoolExecutor(parallel)
    futs = [ex.submit(run, c, None, None, None, False) for c in cmds]
    try:
        return [f.result() for f in futs]
    except KeyboardInterrupt:
        _kill_all(); ex.shutdown(wait=False, cancel_futures=True); raise
    finally:
        stop.set()

def _kill_all():
    with _LOCK:
        for p in list(_PROCS):
            try: os.killpg(p.pid, signal.SIGTERM)
            except Exception: pass
    print('Interrupted - running jobs were stopped; they restart from the beginning next time.', flush=True)

def status():
    done = sorted(f[:-3] for f in os.listdir(DONE) if f.endswith('.ok'))
    print('finished jobs:', len(done)); print(', '.join(done))
print('job runner ready')

job runner ready


In [ ]:
#@title Choose the job groups to run
A_main_rep0 = True  #@param {type:"boolean"}
B_classical_filters = True  #@param {type:"boolean"}
C_prefilter_controls = True  #@param {type:"boolean"}
D_superpixel_sensitivity = True  #@param {type:"boolean"}
E_super_resolution = True  #@param {type:"boolean"}
F_repeated_cv_rep1_rep2 = True  #@param {type:"boolean"}
G_detection_faster_rcnn = True  #@param {type:"boolean"}
H_sam_medsam = True  #@param {type:"boolean"}
I_timing = True  #@param {type:"boolean"}
FORCE_RERUN = False  #@param {type:"boolean"}
if FORCE_RERUN:
    for f in os.listdir(DONE): os.remove(f'{DONE}/{f}')
F4 = range(4)
GROUPS = [
    ('A main comparison, repetition 0', A_main_rep0, [f'python run_seg.py 0 {f}' for f in F4]),
    ('B classical filters',             B_classical_filters, ['python precompute_filters.py']),
    ('C pre-filter controls',           C_prefilter_controls, [f'python run_seg.py 0 {f} --only=pre' for f in F4]),
    ('D superpixel sensitivity',        D_superpixel_sensitivity, [f'python run_seg.py 0 {f} --only=sp' for f in F4]),
    ('E super-resolution fidelity',     E_super_resolution, [f'python run_seg.py 0 {f} --only=sr' for f in F4]),
    ('F repeated CV (rep 1, 2)',        F_repeated_cv_rep1_rep2, [f'python run_seg.py {r} {f}' for r in (1, 2) for f in F4]),
    ('G Faster R-CNN detection',        G_detection_faster_rcnn, [f'python run_det.py {f}' for f in F4]),
    ('H SAM / MedSAM',                  H_sam_medsam, [f'python run_sam.py {k} 2' for k in range(2)]),
    ('I timing',                        I_timing, ['python run_timing.py']),
]
t_all = time.time()
for title, on, cmds in GROUPS:
    if not on: continue
    print(f'\n######## {title} ({len(cmds)} jobs)', flush=True)
    rcs = run_group(cmds, parallel=1 if len(cmds) == 1 else None)
    if any(rcs):
        print(f'!! {sum(1 for r in rcs if r)} job(s) failed in group "{title}" - fix them (Appendix) before running later groups.')
        if title[0] in 'AB': break
print('\nqueue finished in %.1f h' % ((time.time() - t_all) / 3600)); status()


######## A main comparison, repetition 0 (4 jobs)
[start] run_seg_0_0  (07:58:10)
[start] run_seg_0_1  (07:58:10)
--- 08:08 | 2 job(s) running
    run_seg_0_0                  08:05:56 pre-processing done 460.1 s
    run_seg_0_1                  08:06:21 pre-processing done 484.5 s
    run_seg_0_2                  ValueError: need at least one array to stack
    run_seg_0_3                  ValueError: need at least one array to stack
--- 08:18 | 2 job(s) running
    run_seg_0_0                  08:16:31 GCN-dual (original features) epochs 74 nodes 376.3 time 125.1
    run_seg_0_1                  08:16:35 GCN-dual (original features) epochs 150 nodes 377.9 time 244.9
    run_seg_0_2                  ValueError: need at least one array to stack
    run_seg_0_3                  ValueError: need at least one array to stack
--- 08:28 | 2 job(s) running
    run_seg_0_0                  08:25:46 GCN without seeds (automatic) epochs 76 nodes 376.3 time 212.7
    run_seg_0_1                 

While the queue runs, a status line is printed every 10 minutes; full logs can be opened from the Colab file browser (left panel) under `WORK/results/joblogs/`. After the queue stops, this cell shows the last line of the most recent logs:

In [ ]:
import glob, os
for f in sorted(glob.glob(f'{WORK}/results/joblogs/*.log'), key=os.path.getmtime)[-6:]:
    lines = open(f, errors='replace').read().strip().splitlines()
    print(f'{os.path.basename(f):35s} | ' + (lines[-1] if lines else ''))

run_det_1.log                       | DET DONE 1 2637.5774369239807
run_det_2.log                       | DET DONE 2 2615.7611870765686
run_det_3.log                       | DET DONE 3 2580.2335431575775
run_sam_1_2.log                     | SAM DONE
run_sam_0_2.log                     | SAM DONE
run_timing.log                      | 9              Operator time to draw scribbles  not measured (no human-operator study; see Sec...


In [ ]:
run('python analyze.py', name='analyze', force=True, nth=multiprocessing.cpu_count())
run('python fig_qual.py 3 4 5 6b 16 17', name='fig_qual', force=True)
run('python fig_schematics.py --fig1', name='fig_schematics', force=True)

In [ ]:
import pandas as pd, glob
from IPython.display import display, Image, Markdown
T = f'{WORK}/results/tables'
pd.set_option('display.max_columns', 30, 'display.width', 250)
for name in ('T_main', 'T_patient_stats', 'T_ablation', 'T_detection'):
    p = f'{T}/{name}.csv'
    if os.path.exists(p):
        df = pd.read_csv(p); df = df[[c for c in df.columns if not c.startswith('_')]]
        display(Markdown(f'**{name}**')); display(df)
for fig in ('Fig08_dice_distribution', 'Fig09_forest_patient_level', 'Fig04_qualitative_segmentation', 'Fig11_robustness'):
    p = f'{WORK}/figures/{fig}.png'
    if os.path.exists(p):
        display(Markdown(f'**{fig}**')); display(Image(p, width=900))
print('all tables :', sorted(os.path.basename(f) for f in glob.glob(f'{T}/*.csv')))
print('all figures:', sorted({os.path.basename(f)[:-4] for f in glob.glob(f"{WORK}/figures/*.png")}))